# DNR Pre-Training Diagnostic Notebook
## IEEE 33-Bus Distribution Network Reconfiguration
### Graph Transformer + Reptile Meta-Learning — Pre-Training Analysis

---

## Purpose
Before training the Graph Transformer (GT) with Reptile meta-learning,
this notebook runs a full diagnostic pipeline to:

1. Load and validate the IEEE 33-bus network data
2. Build a **realistic physics-guided load sampler** (time-of-day, seasonal, EV, PV)
3. Run **2000 diagnostic scenarios** through the exact BFS solver
4. Analyze **topology class distribution** — how many classes, how balanced
5. Check **loss function scales** — CE vs physics loss alignment
6. Decide **training hyperparameters** based on diagnostic results
7. Store all diagnostic data for reference during training

## Methods
- **BFS Solver**: Backward-Forward Sweep, Baran & Wu (1989)
- **Load Modeling**: Time-series daily curves per bus type, Lotfi & Khodaei (2016)
- **Spatial Correlation**: Gaussian copula, Iman & Conover (1982)
- **OU Noise**: Physics-guided stochastic load variation, Mena et al. (2022)
- **Topology Enumeration**: Spanning tree validation, Merlin & Back (1975)

## References
- [1] Baran & Wu (1989). IEEE Trans. Power Delivery, 4(2), 1401-1407
- [2] Civanlar et al. (1988). IEEE Trans. Power Delivery, 3(3), 1217-1223
- [3] Merlin & Back (1975). Proc. PSCC, 5(1)
- [4] Lotfi & Khodaei (2016). IEEE Trans. Smart Grid, 8(1), 296-304
- [5] Iman & Conover (1982). Communications in Statistics, 11(3), 311-334
- [6] Mena et al. (2022). Energies, 15(7), 2495
- [7] Finn et al. MAML (2017). ICML 2017
- [8] Nichol et al. Reptile (2018). arXiv:1803.02999
- [9] Owerko et al. (2020). ICASSP 2020


In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — IMPORTS
# All dependencies for IEEE 33-bus DNR diagnostic pipeline
# ═══════════════════════════════════════════════════════════════════
import os, time, warnings, copy
from collections import Counter
from itertools import combinations
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import numba as nb
from scipy.stats import qmc
from scipy.special import erfinv
from scipy.stats import norm as sp_norm
from scipy.linalg import cholesky

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

# ── Optional torch import (needed for loss scale check only) ──
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_OK = True
    print(f'PyTorch      : {torch.__version__}')
except ImportError:
    TORCH_OK = False
    print('PyTorch not found — loss scale check will be skipped')

print(f'NumPy        : {np.__version__}')
print(f'Pandas       : {pd.__version__}')
print(f'Numba        : {nb.__version__}')
print('All imports OK.')


PyTorch      : 2.6.0+cu124
NumPy        : 2.3.0
Pandas       : 2.3.3
Numba        : 0.63.1
All imports OK.


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# All hyperparameters in one place.
# Change EXCEL_FILE to match your local filename.
# ═══════════════════════════════════════════════════════════════════

# ── File paths ────────────────────────────────────────────────────
EXCEL_FILE  = 'Data_33bus.xlsx'       # IEEE 33-bus input data
DIAG_FILE   = 'dnr_diagnostic.h5'    # stores diagnostic results
DIAG_N      = 2000                   # scenarios for diagnostic run

# ── Network parameters (IEEE 33-bus standard) ─────────────────────
VN_KV    = 12.66    # nominal voltage kV
BASE_MVA = 10.0     # system base MVA
N_BUS    = 33       # total buses including slack
N_LINES  = 37       # total lines including 5 tie lines
N_OPEN   = 5        # lines open = N_LINES - (N_BUS - 1) = 5

# ── Power flow constraints ─────────────────────────────────────────
# IEEE 1547-2018 voltage bounds
V_MIN_PU = 0.90
V_MAX_PU = 1.05

# Power factor minimum (IEEE 1547)
PF_MIN          = 0.70
Q_BUS_MAX_RATIO = np.tan(np.arccos(PF_MIN))  # ≈ 1.020

# BFS solver settings
BFS_MAX_ITER = 30
BFS_TOL      = 1e-7

# ── Topology ranking ───────────────────────────────────────────────
TOP_K = 50   # top candidates passed from Stage1 to Stage2

# ── Impedance base ────────────────────────────────────────────────
# Z_base = Vbase² / Sbase = 12.66² / 10.0 = 16.03 Ω
Z_BASE = (VN_KV ** 2) / BASE_MVA

print('═'*55)
print('Configuration')
print('═'*55)
print(f'  Network      : IEEE {N_BUS}-bus, {N_LINES} lines, {N_OPEN} tie lines')
print(f'  Voltage      : {V_MIN_PU} – {V_MAX_PU} pu  [IEEE 1547]')
print(f'  PF minimum   : {PF_MIN}  (Q/P ≤ {Q_BUS_MAX_RATIO:.3f})')
print(f'  Z_base       : {Z_BASE:.4f} Ω')
print(f'  BFS          : max_iter={BFS_MAX_ITER}, tol={BFS_TOL}')
print(f'  TOP_K        : {TOP_K}')
print(f'  Diagnostic N : {DIAG_N:,} scenarios')
print(f'  Output       : {DIAG_FILE}')
print('═'*55)


═══════════════════════════════════════════════════════
Configuration
═══════════════════════════════════════════════════════
  Network      : IEEE 33-bus, 37 lines, 5 tie lines
  Voltage      : 0.9 – 1.05 pu  [IEEE 1547]
  PF minimum   : 0.7  (Q/P ≤ 1.020)
  Z_base       : 16.0276 Ω
  BFS          : max_iter=30, tol=1e-07
  TOP_K        : 50
  Diagnostic N : 2,000 scenarios
  Output       : dnr_diagnostic.h5
═══════════════════════════════════════════════════════


In [13]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — LOAD IEEE 33-BUS NETWORK DATA
#
# Reads line and bus data from Excel file.
# Converts impedances from Ohms to per-unit.
# Computes base loads and power factor ratios.
#
# IEEE 33-bus feeder:
#   - 33 buses, 37 lines (32 section lines + 5 tie lines)
#   - Tie lines: 33,34,35,36,37 (normally open)
#   - Substation at bus 1 (slack bus, index 0)
#   - Total base load: ~3.715 MW, ~2.300 Mvar
#
# Ref: Baran & Wu (1989) [1]
# ═══════════════════════════════════════════════════════════════════



# ── Replace this in Cell 3 ────────────────────────────────
# Instead of using raw Q from Excel (has bad PF on some buses)
# Recompute Q from P using per-bus PF capped at 0.85 minimum






line_data = pd.read_excel(EXCEL_FILE, sheet_name='Linedata')
bus_data  = pd.read_excel(EXCEL_FILE, sheet_name='Busdata')

# ── Line connectivity (0-indexed) ─────────────────────────────────
FROM_BUS = (line_data['From bus'].values - 1).astype(np.int32)
TO_BUS   = (line_data['To bus'].values   - 1).astype(np.int32)

# ── Impedance in Ohms (raw from Excel) ────────────────────────────
R_OHM = line_data['r'].values.astype(np.float64)
X_OHM = line_data['x'].values.astype(np.float64)

# ── Convert to per-unit (critical for BFS convergence) ────────────
# Without pu conversion, voltage drops are enormous → BFS diverges
R_PU = R_OHM / Z_BASE
X_PU = X_OHM / Z_BASE

# ── Base loads (MW / Mvar) — buses 2..33, skip slack ──────────────
base_p = bus_data['P_load'].values[1:].astype(np.float64) / 1000.0
base_q = bus_data['Q_load'].values[1:].astype(np.float64) / 1000.0

# Fix power factor: cap Q so PF >= 0.85 per bus
pf_raw    = base_p / (np.sqrt(base_p**2 + base_q**2) + 1e-9)
pf_capped = np.maximum(pf_raw, 0.85)
base_q    = base_p * np.tan(np.arccos(pf_capped))
pf_ratio  = base_q / (base_p + 1e-9)

print('Per-bus power factors after fix:')
pf_check = base_p / (np.sqrt(base_p**2 + base_q**2) + 1e-9)
print(f'  Min PF : {pf_check.min():.3f}  (should be >= 0.85)')
print(f'  Max PF : {pf_check.max():.3f}')
print(f'  Base Q total: {base_q.sum():.4f} Mvar')
# ── Per-bus load bounds ────────────────────────────────────────────
# 30% to 220% of base load covers:
#   light: off-peak / mild weather
#   heavy: peak demand / EV charging / cold snap
# Ref: Lotfi & Khodaei (2016) [4]
P_BUS_MIN = base_p * 0.30
P_BUS_MAX = base_p * 2.20

# ── System total bounds ────────────────────────────────────────────
BASE_TOTAL_P = base_p.sum()
BASE_TOTAL_Q = base_q.sum()
TOTAL_P_MIN  = BASE_TOTAL_P * 0.40
TOTAL_P_MAX  = BASE_TOTAL_P * 2.00

def clip_to_constraints(P, Q):
    """Clip P and Q arrays to physical constraint bounds."""
    P = np.clip(P, P_BUS_MIN, P_BUS_MAX)
    Q = np.clip(Q, 0.0, None)
    Q = np.minimum(Q, P * Q_BUS_MAX_RATIO)
    return P, Q

def check_constraints(p, q):
    """Return (ok:bool, reason:str) for a single scenario."""
    if np.any(p < P_BUS_MIN):           return False, 'P below min'
    if np.any(p > P_BUS_MAX):           return False, 'P above max'
    if np.any(q < 0):                   return False, 'negative Q'
    if np.any(q > p*Q_BUS_MAX_RATIO):  return False, 'PF < 0.70'
    tp = p.sum()
    if tp < TOTAL_P_MIN:                return False, 'total P low'
    if tp > TOTAL_P_MAX:                return False, 'total P high'
    return True, 'ok'

print('═'*55)
print('IEEE 33-Bus Network Data Loaded')
print('═'*55)
print(f'  Lines        : {N_LINES}  (32 section + 5 tie)')
print(f'  R_PU range   : {R_PU.min():.5f} – {R_PU.max():.5f} pu')
print(f'  X_PU range   : {X_PU.min():.5f} – {X_PU.max():.5f} pu')
print(f'  Base total P : {BASE_TOTAL_P:.4f} MW')
print(f'  Base total Q : {BASE_TOTAL_Q:.4f} Mvar')
print(f'  P_BUS_MIN    : {P_BUS_MIN.min():.5f} MW')
print(f'  P_BUS_MAX    : {P_BUS_MAX.max():.5f} MW')
print()
print('Line Data:')
display(line_data)
print()
print('Bus Data:')
display(bus_data)


Per-bus power factors after fix:
  Min PF : 0.850  (should be >= 0.85)
  Max PF : 0.986
  Base Q total: 1.8078 Mvar
═══════════════════════════════════════════════════════
IEEE 33-Bus Network Data Loaded
═══════════════════════════════════════════════════════
  Lines        : 37  (32 section + 5 tie)
  R_PU range   : 0.00575 – 0.12479 pu
  X_PU range   : 0.00293 – 0.12479 pu
  Base total P : 3.7150 MW
  Base total Q : 1.8078 Mvar
  P_BUS_MIN    : 0.01350 MW
  P_BUS_MAX    : 0.92400 MW

Line Data:


,Line Number,From bus,To bus,r,x
0,1,1,2,0.0922,0.0470
1,2,2,3,0.4930,0.2512
2,3,3,4,0.3661,0.1864
3,4,4,5,0.3811,0.1941
4,5,5,6,0.8190,0.7070
5,6,6,7,0.1872,0.6188
6,7,7,8,0.7115,0.2351
7,8,8,9,1.0299,0.7400
8,9,9,10,1.0440,0.7400
9,10,10,11,0.1967,0.0651



Bus Data:


,Bus No.,P_load,Q_load,Unnamed: 3
0,1,0,0,PV
1,2,100,60,PQ
2,3,90,40,PQ
3,4,120,80,PQ
4,5,60,30,PQ
5,6,60,20,PQ
6,7,200,100,PQ
7,8,200,100,PQ
8,9,60,20,PQ
9,10,60,20,PQ


In [14]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — SPATIAL CORRELATION (GAUSSIAN COPULA)
#
# Real distribution feeders show strong spatial correlation:
# buses on the same lateral tend to load/unload together
# (e.g. morning peak hits all residential buses simultaneously).
#
# We model this with a Gaussian copula using intra-feeder
# correlation = 0.80, adjacent groups = 0.45, cross = 0.15.
#
# Ref: Iman & Conover (1982) [5]
# ═══════════════════════════════════════════════════════════════════

# ── IEEE 33-bus lateral groups (0-indexed bus numbers) ────────────
# Based on the physical feeder topology from Baran & Wu (1989) [1]
LATERAL_GROUPS = {
    'main_near' : list(range(0,  9)),    # buses 1-9   (main feeder near)
    'main_mid'  : list(range(9,  18)),   # buses 10-18 (main feeder mid)
    'lateral1'  : [2, 19, 20, 21, 22],  # branch from bus 3
    'lateral2'  : [8, 23, 24, 25],      # branch from bus 9
    'lateral3'  : list(range(25, 32)),   # buses 26-32 (far lateral)
}

def build_correlation_matrix(n=32):
    """
    Build 32x32 spatial correlation matrix for bus loads.
    Intra-group = 0.80 (buses on same lateral move together)
    Adjacent    = 0.45 (neighboring laterals moderately correlated)
    Cross       = 0.15 (far apart buses weakly correlated)
    """
    C    = np.eye(n)
    grps = list(LATERAL_GROUPS.values())

    # Intra-group correlation
    for g in grps:
        for i in g:
            for j in g:
                if i != j:
                    C[i, j] = 0.80

    # Adjacent group correlation
    adj_pairs = [(0,1),(1,2),(1,3),(2,3),(3,4)]
    for gi, gj in adj_pairs:
        for i in grps[gi]:
            for j in grps[gj]:
                if C[i, j] == 0:
                    C[i, j] = 0.45

    # Cross-feeder (remaining zeros → 0.15)
    C[C == 0] = 0.15
    np.fill_diagonal(C, 1.0)

    # Ensure positive definite
    ev = np.linalg.eigvalsh(C)
    if ev.min() < 1e-6:
        C += np.eye(n) * (abs(ev.min()) + 1e-4)
        D  = np.diag(1 / np.sqrt(np.diag(C)))
        C  = D @ C @ D

    return C

CORR_MATRIX = build_correlation_matrix()
CHOL_L      = np.linalg.cholesky(CORR_MATRIX)

min_ev = np.linalg.eigvalsh(CORR_MATRIX).min()
print(f'Correlation matrix: 32×32')
print(f'Min eigenvalue    : {min_ev:.6f}  (>0 = positive definite ✓)')
print(f'Cholesky factor   : {CHOL_L.shape}')

# Quick sanity: diagonal should be 1.0
assert np.allclose(np.diag(CORR_MATRIX), 1.0), 'Diagonal not 1.0!'
print('Diagonal check    : ✓')


Correlation matrix: 32×32
Min eigenvalue    : 0.000065  (>0 = positive definite ✓)
Cholesky factor   : (32, 32)
Diagonal check    : ✓


In [15]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — BUS TYPE CLASSIFICATION & DAILY LOAD CURVES
#
# Real distribution feeders serve mixed load types.
# Each type has a distinct daily load profile shape.
# This is the foundation of the realistic load sampler.
#
# Bus classification based on:
#   - Proximity to substation (near = commercial/mixed)
#   - Lateral branch type (residential laterals)
#   - Load magnitude in base case
#
# Daily curves normalized to 1.0 at peak.
# 24 hourly values: index 0 = midnight, index 12 = noon.
#
# Ref: Lotfi & Khodaei (2016) [4]
# ═══════════════════════════════════════════════════════════════════

# ── Bus type assignment (1-indexed bus numbers) ────────────────────
BUS_TYPES = {
    'residential' : [1,2,3,4,5,12,13,14,15,16,17,18,19,20,21,22],
    'commercial'  : [6,7,8,9,10,11,23,24,25],
    'industrial'  : [26,27,28,29,30,31,32]
}

# ── Hourly load curves (24h, normalized to peak=1.0) ──────────────
# Residential: dual peak (morning 7-9am, evening 6-9pm)
# Commercial : business hours peak (9am-6pm)
# Industrial : flat during working hours
DAILY_CURVES = {
    'residential': np.array([
        0.40,0.32,0.30,0.30,0.35,0.55,   # 0-5  (midnight to 5am)
        0.75,1.00,0.90,0.80,0.72,0.70,   # 6-11 (morning peak 7am)
        0.72,0.75,0.78,0.85,0.95,1.20,   # 12-17 (afternoon rise)
        1.10,1.00,0.90,0.80,0.65,0.50,   # 18-23 (evening peak 17-18h)
    ]),
    'commercial': np.array([
        0.20,0.20,0.20,0.20,0.22,0.30,
        0.50,0.78,1.00,1.00,1.00,0.98,
        0.90,1.00,1.00,1.00,0.92,0.75,
        0.55,0.35,0.28,0.22,0.20,0.20,
    ]),
    'industrial': np.array([
        0.70,0.70,0.70,0.70,0.72,0.80,
        0.90,1.00,1.00,1.00,1.00,1.00,
        1.00,1.00,1.00,1.00,1.00,0.92,
        0.82,0.75,0.72,0.70,0.70,0.70,
    ]),
}

# Normalize all curves to [0,1] with peak = 1.0
for k in DAILY_CURVES:
    DAILY_CURVES[k] /= DAILY_CURVES[k].max()

# ── Seasonal multipliers ───────────────────────────────────────────
# Winter: heating load increases demand
# Summer: cooling (AC) increases demand
# Spring/Autumn: moderate
SEASONAL_SCALE = {
    'winter': 1.25,
    'summer': 1.15,
    'spring': 0.90,
    'autumn': 0.95,
}
SEASONS = list(SEASONAL_SCALE.keys())

# ── EV charging parameters ─────────────────────────────────────────
# Evening charging spike on residential buses (6pm-10pm)
# Models increasing EV penetration in modern distribution grids
EV_BUSES      = [b-1 for b in [1,2,3,4,12,13,18,19,20]]  # 0-indexed
EV_PEAK_HOURS = {18, 19, 20, 21}
EV_EXTRA_LOAD = 0.30   # up to 30% extra load from EV charging

# ── Solar PV parameters ────────────────────────────────────────────
# Midday net load reduction on PV-equipped buses (10am-2pm)
# PV generation offsets demand → negative injection modeled as
# reduced net load (conservative: floor at 10% of base)
PV_BUSES      = [b-1 for b in [4,5,6,13,14,22,23]]  # 0-indexed
PV_PEAK_HOURS = {10, 11, 12, 13, 14}
PV_REDUCTION  = 0.25   # up to 25% load reduction at peak PV

# ── OU noise parameters per bus type ──────────────────────────────
# Industrial loads most stable (large inertia, contracted demand)
# Residential most variable (many small independent consumers)
# Ref: Mena et al. (2022) [6]
NOISE_PARAMS = {
    'residential': {'sigma': 0.08, 'theta': 0.30},
    'commercial' : {'sigma': 0.05, 'theta': 0.40},
    'industrial' : {'sigma': 0.03, 'theta': 0.50},
}

print('Bus Type Classification:')
print('─'*45)
for btype, buses in BUS_TYPES.items():
    print(f'  {btype:12s}: buses {buses}')
print()
print('Daily curve peaks (hour of max load):')
for btype, curve in DAILY_CURVES.items():
    print(f'  {btype:12s}: peak at hour {np.argmax(curve):02d}:00  '
          f'min={curve.min():.2f}  max={curve.max():.2f}')
print()
print('Seasonal multipliers:', SEASONAL_SCALE)
print(f'EV buses (0-idx)    : {EV_BUSES}')
print(f'PV buses (0-idx)    : {PV_BUSES}')


Bus Type Classification:
─────────────────────────────────────────────
  residential : buses [1, 2, 3, 4, 5, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
  commercial  : buses [6, 7, 8, 9, 10, 11, 23, 24, 25]
  industrial  : buses [26, 27, 28, 29, 30, 31, 32]

Daily curve peaks (hour of max load):
  residential : peak at hour 17:00  min=0.25  max=1.00
  commercial  : peak at hour 08:00  min=0.20  max=1.00
  industrial  : peak at hour 07:00  min=0.70  max=1.00

Seasonal multipliers: {'winter': 1.25, 'summer': 1.15, 'spring': 0.9, 'autumn': 0.95}
EV buses (0-idx)    : [0, 1, 2, 3, 11, 12, 17, 18, 19]
PV buses (0-idx)    : [3, 4, 5, 12, 13, 21, 22]


In [16]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — REALISTIC PHYSICS-GUIDED LOAD SAMPLER
#
# Generates load scenarios that reflect real operating conditions
# of the IEEE 33-bus feeder. Five layers of realism:
#
# Layer 1: Daily curve per bus type (time-of-day pattern)
# Layer 2: Seasonal scaling (winter/summer/spring/autumn)
# Layer 3: EV charging spike (evening residential buses)
# Layer 4: Solar PV reduction (midday PV-equipped buses)
# Layer 5: Spatially correlated OU noise (realistic variability)
#
# Also provides topology-steered and stress scenario samplers
# for training diversity.
# ═══════════════════════════════════════════════════════════════════

def sample_realistic_scenario(episode=0, rng=None):
    """
    Generate one realistic load scenario for IEEE 33-bus.

    Returns:
        P_full : (33,) active power per bus [MW], index 0 = slack = 0
        Q_full : (33,) reactive power per bus [Mvar]
        ctx    : dict with time context (hour, season, weekend, ev, pv)
    """
    if rng is None:
        rng = np.random.default_rng(episode)

    # ── Layer 1: Time context ──────────────────────────────────────
    hour       = int(rng.integers(0, 24))
    season     = SEASONS[int(rng.integers(0, 4))]
    is_weekend = rng.random() < 0.286   # ~2/7 days

    # ── Layer 2: Daily curve per bus type ─────────────────────────
    P = np.zeros(32)
    for btype, buses in BUS_TYPES.items():
        curve_val = DAILY_CURVES[btype][hour]
        # Weekend: commercial drops 40%, others slight drop
        if is_weekend:
            if btype == 'commercial': curve_val *= 0.60
            elif btype == 'residential': curve_val *= 1.05  # home all day
        for b in buses:
            P[b-1] = base_p[b-1] * curve_val

    # ── Layer 3: Seasonal scaling ──────────────────────────────────
    P *= SEASONAL_SCALE[season]

    # ── Layer 4: EV charging spike ─────────────────────────────────
    ev_active = hour in EV_PEAK_HOURS
    if ev_active:
        for b in EV_BUSES:
            if b < 32:
                ev_factor = 1.0 + EV_EXTRA_LOAD * rng.random()
                P[b]     *= ev_factor

    # ── Layer 5: Solar PV reduction ────────────────────────────────
    pv_active = (hour in PV_PEAK_HOURS) and (season in ['spring','summer'])
    if pv_active:
        for b in PV_BUSES:
            if b < 32:
                pv_factor = 1.0 - PV_REDUCTION * rng.random()
                P[b]     *= pv_factor
                P[b]      = max(P[b], base_p[b] * 0.10)

    # ── Layer 6: Spatially correlated OU noise ─────────────────────
    # Correlated noise via Cholesky decomposition of spatial
    # correlation matrix. Each bus type has own noise level.
    noise_vec = np.zeros(32)
    for btype, buses in BUS_TYPES.items():
        sigma = NOISE_PARAMS[btype]['sigma']
        raw   = rng.normal(0, sigma, 32)
        corr  = CHOL_L @ raw   # spatial correlation
        for b in buses:
            noise_vec[b-1] = corr[b-1]

    P *= (1.0 + noise_vec)

    # ── Compute Q from power factor ratio ─────────────────────────
    Q = P * pf_ratio

    # ── Apply physical constraints ────────────────────────────────
    P, Q = clip_to_constraints(P, Q)

    # ── Add slack bus (index 0, always zero load) ─────────────────
    P_full = np.concatenate([[0.0], P])
    Q_full = np.concatenate([[0.0], Q])

    ctx = {
        'hour': hour, 'season': season,
        'weekend': is_weekend,
        'ev_active': ev_active, 'pv_active': pv_active
    }
    return P_full, Q_full, ctx


# ── Topology-steered sampler ───────────────────────────────────────
# 25 profiles that bias loads toward specific feeder sections
# to ensure all topology classes appear in training.
# Ref: Civanlar et al. (1988) [2]
TOPO_PROFILES = [
    {'mn':0.55,'mm':0.55,'l1':0.55,'l2':0.55,'l3':0.55,'std':0.07},
    {'mn':1.80,'mm':1.80,'l1':1.80,'l2':1.80,'l3':1.80,'std':0.07},
    {'mn':1.70,'mm':0.75,'l1':0.75,'l2':0.75,'l3':0.75,'std':0.08},
    {'mn':0.75,'mm':1.70,'l1':0.75,'l2':0.75,'l3':0.75,'std':0.08},
    {'mn':0.75,'mm':0.75,'l1':1.90,'l2':0.75,'l3':0.75,'std':0.08},
    {'mn':0.75,'mm':0.75,'l1':0.75,'l2':1.90,'l3':0.75,'std':0.08},
    {'mn':0.75,'mm':0.75,'l1':0.75,'l2':0.75,'l3':1.90,'std':0.08},
    {'mn':1.60,'mm':1.60,'l1':0.65,'l2':0.65,'l3':0.65,'std':0.09},
    {'mn':0.65,'mm':0.65,'l1':1.60,'l2':1.60,'l3':1.60,'std':0.09},
    {'mn':1.50,'mm':0.80,'l1':1.50,'l2':0.80,'l3':0.80,'std':0.09},
    {'mn':0.80,'mm':1.50,'l1':0.80,'l2':1.50,'l3':0.80,'std':0.09},
    {'mn':0.80,'mm':0.80,'l1':0.80,'l2':1.50,'l3':1.50,'std':0.09},
    {'mn':1.40,'mm':1.40,'l1':0.70,'l2':1.40,'l3':0.70,'std':0.09},
    {'mn':1.00,'mm':1.90,'l1':1.00,'l2':0.60,'l3':1.90,'std':0.09},
    {'mn':1.90,'mm':1.00,'l1':0.60,'l2':1.90,'l3':1.00,'std':0.09},
    {'mn':0.60,'mm':1.80,'l1':1.80,'l2':0.60,'l3':1.80,'std':0.09},
    {'mn':1.80,'mm':0.60,'l1':1.80,'l2':1.80,'l3':0.60,'std':0.09},
    {'mn':1.20,'mm':1.20,'l1':1.20,'l2':1.20,'l3':1.20,'std':0.06},
    {'mn':0.90,'mm':1.50,'l1':1.50,'l2':0.90,'l3':0.90,'std':0.10},
    {'mn':1.50,'mm':0.90,'l1':0.90,'l2':0.90,'l3':1.50,'std':0.10},
    {'mn':1.10,'mm':1.10,'l1':0.70,'l2':1.70,'l3':1.10,'std':0.10},
    {'mn':1.10,'mm':1.70,'l1':1.10,'l2':1.10,'l3':0.70,'std':0.10},
    {'mn':1.70,'mm':1.10,'l1':1.10,'l2':0.70,'l3':1.10,'std':0.10},
    {'mn':0.70,'mm':1.10,'l1':1.70,'l2':1.10,'l3':1.10,'std':0.10},
    {'mn':1.10,'mm':0.70,'l1':1.10,'l2':1.10,'l3':1.70,'std':0.10},
]

def sample_steered(profile_idx=0, rng=None):
    """Generate load scenario steered toward a specific topology profile."""
    if rng is None:
        rng = np.random.default_rng(profile_idx)
    prof  = TOPO_PROFILES[profile_idx % len(TOPO_PROFILES)]
    grps  = LATERAL_GROUPS
    ms    = np.ones(32)
    keys  = ['main_near','main_mid','lateral1','lateral2','lateral3']
    mkeys = ['mn','mm','l1','l2','l3']
    for key, mkey in zip(keys, mkeys):
        for i in grps[key]:
            if i < 32: ms[i] = prof[mkey]
    noise = rng.normal(0, prof['std'], 32)
    scale = np.clip(ms + noise, 0.30, 2.20)
    P     = base_p * scale
    Q     = P * pf_ratio
    P, Q  = clip_to_constraints(P, Q)
    P_full = np.concatenate([[0.0], P])
    Q_full = np.concatenate([[0.0], Q])
    return P_full, Q_full, {'hour': -1, 'season': 'steered',
                             'weekend': False,
                             'ev_active': False, 'pv_active': False}

def sample_stress(mode='heavy', rng=None, seed=0):
    """Generate extreme load scenarios (very light or very heavy)."""
    if rng is None:
        rng = np.random.default_rng(seed)
    if mode == 'heavy':
        scale = rng.uniform(1.80, 2.20, 32)
    else:
        scale = rng.uniform(0.30, 0.50, 32)
    P = np.clip(base_p * scale, P_BUS_MIN, P_BUS_MAX)
    Q = np.clip(P * pf_ratio, 0, P * Q_BUS_MAX_RATIO)
    P_full = np.concatenate([[0.0], P])
    Q_full = np.concatenate([[0.0], Q])
    return P_full, Q_full, {'hour': -1, 'season': mode,
                             'weekend': False,
                             'ev_active': False, 'pv_active': False}

def sample_for_episode(episode):
    """
    Episode mixer — combines all samplers for training diversity.
    60% realistic | 20% topology steered | 20% stress
    """
    rng = np.random.default_rng(episode)
    r   = rng.random()
    if r < 0.60:
        return sample_realistic_scenario(episode, rng)
    elif r < 0.80:
        return sample_steered(episode % len(TOPO_PROFILES), rng)
    else:
        mode = 'heavy' if rng.random() < 0.5 else 'light'
        return sample_stress(mode, rng, episode)

# ── Quick validation ───────────────────────────────────────────────
print('Sampler validation (5 scenarios):')
print('─'*55)
for i in range(5):
    P, Q, ctx = sample_realistic_scenario(i)
    ok, reason = check_constraints(P[1:], Q[1:])
    print(f'  Scenario {i}: hour={ctx["hour"]:02d}h '
          f'season={ctx["season"]:6s} '
          f'EV={ctx["ev_active"]} PV={ctx["pv_active"]}')
    print(f'    TotalP={P[1:].sum():.3f}MW  '
          f'TotalQ={Q[1:].sum():.3f}Mvar  '
          f'constraints={ok} ({reason})')


Sampler validation (5 scenarios):
───────────────────────────────────────────────────────
  Scenario 0: hour=20h season=spring EV=True PV=False
    TotalP=2.031MW  TotalQ=0.890Mvar  constraints=True (ok)
  Scenario 1: hour=11h season=spring EV=False PV=True
    TotalP=2.683MW  TotalQ=1.313Mvar  constraints=True (ok)
  Scenario 2: hour=20h season=summer EV=True PV=False
    TotalP=2.348MW  TotalQ=1.149Mvar  constraints=True (ok)
  Scenario 3: hour=19h season=winter EV=True PV=False
    TotalP=2.801MW  TotalQ=1.341Mvar  constraints=True (ok)
  Scenario 4: hour=17h season=autumn EV=False PV=False
    TotalP=3.101MW  TotalQ=1.506Mvar  constraints=True (ok)


In [17]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — ENUMERATE VALID RADIAL TOPOLOGIES
#
# A valid DNR topology must be a spanning tree of the 33-bus graph:
#   - All 33 buses connected (radial = no loops)
#   - Exactly 32 lines active (N_BUS - 1)
#   - Exactly 5 lines open (the tie lines or others)
#
# We enumerate all C(37,5) = 435,897 combinations and filter
# to radial ones using BFS connectivity check.
# Result: ~50,751 valid spanning trees.
#
# Ref: Merlin & Back (1975) [3]
# ═══════════════════════════════════════════════════════════════════

def is_radial(from_b, to_b, open_set, n_bus=33):
    """
    Check if removing open_set lines leaves a connected spanning tree.
    Uses BFS from slack bus (0). Returns True if all buses reachable.
    """
    adj = [[] for _ in range(n_bus)]
    for k in range(len(from_b)):
        if k not in open_set:
            adj[from_b[k]].append(to_b[k])
            adj[to_b[k]].append(from_b[k])
    visited = np.zeros(n_bus, dtype=bool)
    queue   = [0]; visited[0] = True; count = 1
    while queue:
        node = queue.pop()
        for nb_ in adj[node]:
            if not visited[nb_]:
                visited[nb_] = True
                count        += 1
                queue.append(nb_)
    return count == n_bus

print('Enumerating valid radial topologies...')
print(f'Total combinations C(37,5) = {len(list(combinations(range(37),5))):,}')
t0 = time.time()

VALID_TOPOS = []
for combo in combinations(range(N_LINES), N_OPEN):
    if is_radial(FROM_BUS, TO_BUS, set(combo)):
        VALID_TOPOS.append(combo)

N_TOPOS    = len(VALID_TOPOS)
TOPO_ARRAY = np.array(VALID_TOPOS, dtype=np.int32)  # (N_TOPOS, 5)

print(f'Valid spanning trees : {N_TOPOS:,}')
print(f'Enumeration time     : {time.time()-t0:.1f}s')
print(f'TOPO_ARRAY shape     : {TOPO_ARRAY.shape}')

# ── Precompute mask matrix for vectorized Stage 1 ──────────────────
# KEEP_MASK[t,k] = True if line k is ACTIVE in topology t
# R_KEEP[t,k]   = R_PU[k] if active, 0 if open
# This enables Stage 1 ranking as a single matrix multiply: R_KEEP @ pq2
print()
print('Precomputing mask matrix for vectorized ranking...')
KEEP_MASK = np.ones((N_TOPOS, N_LINES), dtype=bool)
for t in range(N_TOPOS):
    for k in TOPO_ARRAY[t]:
        KEEP_MASK[t, k] = False

R_KEEP = R_PU[None, :] * KEEP_MASK   # (N_TOPOS, 37) per-unit

print(f'KEEP_MASK shape : {KEEP_MASK.shape}  ({KEEP_MASK.nbytes/1e6:.1f} MB)')
print(f'R_KEEP shape    : {R_KEEP.shape}  ({R_KEEP.nbytes/1e6:.1f} MB)')


Enumerating valid radial topologies...
Total combinations C(37,5) = 435,897
Valid spanning trees : 50,751
Enumeration time     : 22.5s
TOPO_ARRAY shape     : (50751, 5)

Precomputing mask matrix for vectorized ranking...
KEEP_MASK shape : (50751, 37)  (1.9 MB)
R_KEEP shape    : (50751, 37)  (15.0 MB)


In [24]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — BACKWARD-FORWARD SWEEP (BFS) POWER FLOW SOLVER
#
# Exact power flow solver for radial distribution networks.
# Uses iterative Backward-Forward Sweep (BFS) method:
#
# Backward sweep: accumulate branch flows leaf → root
#   P_k = P_load[to_k] + sum(P_children) + R_k * I²_k
#   Q_k = Q_load[to_k] + sum(Q_children) + X_k * I²_k
#
# Forward sweep: update voltages root → leaf
#   V²[to_k] = V²[from_k] - 2(R_k*P_k + X_k*Q_k) + Z²_k * I²_k
#
# Convergence: max|V² - V²_old| < tol
#
# Impedances are in per-unit. Loads enter as MW/Mvar and are
# converted to per-unit on BASE_MVA inside the solver.
#
# Ref: Baran & Wu (1989) [1]; Das (2002) Electrical Power Systems
# ═══════════════════════════════════════════════════════════════════

@nb.njit(cache=True)
def bfs_solve(from_bus, to_bus, r, x, p_load, q_load,
              v0_sq=1.0, max_iter=30, tol=1e-7, base_mva=10.0):
    n_lines = len(from_bus)
    n_bus   = len(p_load)

    # Loads are stored in MW/Mvar in the notebook; DistFlow equations need pu.
    p_pu = p_load / base_mva
    q_pu = q_load / base_mva

    # Root the active undirected radial topology at the slack bus. Active tie
    # branches may be listed opposite to the actual slack-to-leaf direction.
    order         = np.zeros(n_lines, dtype=nb.int32)
    oriented_from = np.zeros(n_lines, dtype=nb.int32)
    oriented_to   = np.zeros(n_lines, dtype=nb.int32)
    visited       = np.zeros(n_bus,   dtype=nb.boolean)
    queue         = np.zeros(n_bus,   dtype=nb.int32)
    visited[0] = True
    queue[0]   = 0
    head = 0; tail = 1; order_tail = 0
    while head < tail:
        bus = queue[head]
        head += 1
        for k in range(n_lines):
            if from_bus[k] == bus:
                nb_bus = to_bus[k]
            elif to_bus[k] == bus:
                nb_bus = from_bus[k]
            else:
                continue
            if not visited[nb_bus]:
                visited[nb_bus]       = True
                queue[tail]           = nb_bus
                tail                 += 1
                oriented_from[k]      = bus
                oriented_to[k]        = nb_bus
                order[order_tail]     = k
                order_tail           += 1

    V2 = np.ones(n_bus) * v0_sq
    Pl = np.zeros(n_lines)
    Ql = np.zeros(n_lines)
    if tail != n_bus or order_tail != n_lines:
        return V2, Pl, Ql, False

    for _ in range(max_iter):
        V2_old = V2.copy()

        # Backward sweep: leaf → root
        Pl_new = np.zeros(n_lines)
        Ql_new = np.zeros(n_lines)
        for idx in range(n_lines - 1, -1, -1):
            k  = order[idx]
            tb = oriented_to[k]
            Pl_new[k] = p_pu[tb]
            Ql_new[k] = q_pu[tb]
            for m in range(n_lines):
                if oriented_from[m] == tb:
                    Pl_new[k] += Pl_new[m]
                    Ql_new[k] += Ql_new[m]
            I2k        = (Pl_new[k]**2 + Ql_new[k]**2) / (V2[oriented_from[k]] + 1e-12)
            Pl_new[k] += r[k] * I2k
            Ql_new[k] += x[k] * I2k
        Pl = Pl_new
        Ql = Ql_new

        # Forward sweep: root → leaf
        for idx in range(n_lines):
            k      = order[idx]
            fb     = oriented_from[k]
            tb     = oriented_to[k]
            I2k    = (Pl[k]**2 + Ql[k]**2) / (V2[fb] + 1e-12)
            V2[tb] = (V2[fb]
                      - 2*(r[k]*Pl[k] + x[k]*Ql[k])
                      + (r[k]**2 + x[k]**2) * I2k)
            if V2[tb] < 1e-3:
                V2[tb] = 1e-3   # soft floor only for numerical safety

        if np.max(np.abs(V2 - V2_old)) < tol:
            return V2, Pl, Ql, True

    return V2, Pl, Ql, False


# Force recompile
print('Recompiling BFS with slack-rooted topology orientation...')
_f = np.array([0,1,2], dtype=np.int32)
_t = np.array([1,2,3], dtype=np.int32)
_r = np.array([0.01,0.01,0.01])
_x = np.array([0.005,0.005,0.005])
_p = np.array([0.0,0.05,0.05,0.05])
_q = np.array([0.0,0.02,0.02,0.02])
bfs_solve(_f, _t, _r, _x, _p, _q)
print('Done ✓')

@nb.njit(cache=True)
def compute_loss(from_bus, to_bus, r, V2, Pl, Ql, base_mva=10.0):
    """Compute total I²R losses in MW from per-unit branch flows."""
    n_lines = len(from_bus)
    n_bus   = len(V2)

    # Match bfs_solve orientation so reversed active branches use the
    # slack-side voltage in the I² denominator.
    parent_bus = np.zeros(n_lines, dtype=nb.int32)
    visited    = np.zeros(n_bus,   dtype=nb.boolean)
    queue      = np.zeros(n_bus,   dtype=nb.int32)
    assigned   = np.zeros(n_lines, dtype=nb.boolean)
    visited[0] = True
    queue[0]   = 0
    head = 0; tail = 1; assigned_count = 0
    while head < tail:
        bus = queue[head]
        head += 1
        for k in range(n_lines):
            if assigned[k]:
                continue
            if from_bus[k] == bus:
                nb_bus = to_bus[k]
            elif to_bus[k] == bus:
                nb_bus = from_bus[k]
            else:
                continue
            if not visited[nb_bus]:
                visited[nb_bus] = True
                queue[tail]     = nb_bus
                tail           += 1
                parent_bus[k]   = bus
                assigned[k]     = True
                assigned_count += 1

    if tail != n_bus or assigned_count != n_lines:
        return np.nan

    loss_pu = 0.0
    for k in range(n_lines):
        I2k     = (Pl[k]**2 + Ql[k]**2) / (V2[parent_bus[k]] + 1e-12)
        loss_pu += r[k] * I2k
    return loss_pu * base_mva


# @nb.njit(cache=True)
# def distflow_loss_approx(from_bus, to_bus, r, p_load, q_load):
#     """
#     Flat-voltage approximation of I²R losses.
#     O(n) per topology — used for Stage 1 ranking.
#     Ref: Baran & Wu (1989) [1] eq. (4)
#     """
#     loss = 0.0
#     for k in range(len(from_bus)):
#         loss += r[k] * (p_load[to_bus[k]]**2 + q_load[to_bus[k]]**2)
#     return loss


# # ── Compile Numba kernels ──────────────────────────────────────────
# print('Compiling Numba kernels (first run ~20s)...')
# _f = np.array([0,1,2], dtype=np.int32)
# _t = np.array([1,2,3], dtype=np.int32)
# _r = np.array([0.01,0.01,0.01])
# _x = np.array([0.005,0.005,0.005])
# _p = np.array([0.0,0.05,0.05,0.05])
# _q = np.array([0.0,0.02,0.02,0.02])
# bfs_solve(_f, _t, _r, _x, _p, _q)
# distflow_loss_approx(_f, _t, _r, _p, _q)
# print('Kernels compiled ✓')

# ── BFS sanity check on default IEEE 33-bus topology ──────────────
# Default: tie lines 33-37 open (0-indexed: 32-36)
print()
print('BFS sanity check on default topology (tie lines 33-37 open):')
open_default = {32,33,34,35,36}
kf = np.array([FROM_BUS[k] for k in range(N_LINES) if k not in open_default], dtype=np.int32)
kt = np.array([TO_BUS[k]   for k in range(N_LINES) if k not in open_default], dtype=np.int32)
kr = np.array([R_PU[k]     for k in range(N_LINES) if k not in open_default])
kx = np.array([X_PU[k]     for k in range(N_LINES) if k not in open_default])
p0 = np.concatenate([[0.0], base_p])
# Use raw IEEE Q for the published default-topology benchmark; Cell 3
# may cap base_q separately for scenario-generation constraints.
q0 = bus_data['Q_load'].values.astype(np.float64) / 1000.0

V2s, Pls, Qls, cvg = bfs_solve(kf, kt, kr, kx, p0, q0)
loss_default        = compute_loss(kf, kt, kr, V2s, Pls, Qls)
vmin_default        = np.sqrt(V2s.min())

print(f'  Converged  : {cvg}')
print(f'  V_min (pu) : {vmin_default:.4f}  (should be ~0.913)')
print(f'  Loss (MW)  : {loss_default:.5f}  (should be ~0.202)')
print()
if not cvg:
    print('WARNING: BFS did not converge on default topology!')
    print('Check R_PU/X_PU values and FROM_BUS/TO_BUS ordering.')
else:
    print('BFS solver working correctly ✓')


Recompiling BFS with adjacency list...
Done ✓

BFS sanity check on default topology (tie lines 33-37 open):
  Converged  : False
  V_min (pu) : nan  (should be ~0.913)
  Loss (MW)  : nan  (should be ~0.202)

Check R_PU/X_PU values and FROM_BUS/TO_BUS ordering.


In [20]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — TWO-STAGE DNR SOLVER
#
# Stage 1: Vectorized DistFlow ranking (all N_TOPOS topologies)
#   - approx_loss = R_KEEP @ pq2  (one matrix multiply)
#   - selects TOP_K candidates
#   - O(N_TOPOS) but vectorized → very fast
#
# Stage 2: Exact BFS verification on TOP_K candidates
#   - runs full iterative BFS power flow
#   - checks voltage constraint V_min >= 0.90 pu
#   - selects minimum loss feasible topology
#
# Ref: Baran & Wu (1989) [1]; Chiang & Jean-Jumeau (1990)
# ═══════════════════════════════════════════════════════════════════
INF_LOSS = 1e15

def solve_one_fast(p_bus, q_bus):
    """
    Two-stage DNR solver.

    Parameters
    ----------
    p_bus : (33,) active power per bus [MW], index 0 = slack
    q_bus : (33,) reactive power per bus [Mvar]

    Returns
    -------
    dict with keys: open_lines (1-indexed), min_loss_mw, v_min_pu
    or None if no feasible topology found.
    """
    p_load = p_bus.astype(np.float64)
    q_load = q_bus.astype(np.float64)

    # ── Stage 1: vectorized DistFlow ranking ──────────────────────
    # pq2[k] = P[to_bus[k]]² + Q[to_bus[k]]²
    pq2         = p_load[TO_BUS]**2 + q_load[TO_BUS]**2
    approx_loss = R_KEEP @ pq2          # (N_TOPOS,) — single matmul
    top_idx     = np.argsort(approx_loss)[:TOP_K]

    # ── Stage 2: exact BFS on top candidates ─────────────────────
    best_loss = INF_LOSS
    best_t    = -1
    best_v2   = None

    for t in top_idx:
        open_set = set(TOPO_ARRAY[t])
        keep_f   = np.array([FROM_BUS[k] for k in range(N_LINES)
                              if k not in open_set], dtype=np.int32)
        keep_t   = np.array([TO_BUS[k]   for k in range(N_LINES)
                              if k not in open_set], dtype=np.int32)
        keep_r   = np.array([R_PU[k]     for k in range(N_LINES)
                              if k not in open_set])
        keep_x   = np.array([X_PU[k]     for k in range(N_LINES)
                              if k not in open_set])

        V2, Pl, Ql, cvg = bfs_solve(keep_f, keep_t, keep_r, keep_x,
                                     p_load, q_load,
                                     max_iter=BFS_MAX_ITER, tol=BFS_TOL)
        if not cvg:
            continue
        v_min = np.sqrt(V2.min())
        if v_min < V_MIN_PU:
            continue   # voltage constraint violated
        loss = compute_loss(keep_f, keep_t, keep_r, V2, Pl, Ql)
        if loss < best_loss:
            best_loss = loss
            best_t    = t
            best_v2   = V2.copy()

    if best_t == -1:
        return None   # no feasible topology found

    open_lines = np.sort(TOPO_ARRAY[best_t]) + 1   # 1-indexed
    return {
        'open_lines'  : open_lines,
        'min_loss_mw' : float(best_loss),
        'v_min_pu'    : float(np.sqrt(best_v2.min())),
    }

# ── Solver sanity check ────────────────────────────────────────────
print('Solver sanity check on base load scenario...')
p_test = np.concatenate([[0.0], base_p])
q_test = np.concatenate([[0.0], base_q])
t0     = time.time()
res    = solve_one_fast(p_test, q_test)
t1     = time.time()
print(f'  Result     : {res}')
print(f'  Time       : {(t1-t0)*1000:.1f} ms')
print()
if res is None:
    print('ERROR: solver returned None on base loads!')
    print('Check BFS solver and R_PU/X_PU conversion.')
else:
    print(f'  Open lines : {list(res["open_lines"])}')
    print(f'  Loss (MW)  : {res["min_loss_mw"]:.5f}')
    print(f'  V_min (pu) : {res["v_min_pu"]:.4f}')
    print('Solver working correctly ✓')


Solver sanity check on base load scenario...
  Result     : None
  Time       : 26.4 ms

ERROR: solver returned None on base loads!
Check BFS solver and R_PU/X_PU conversion.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — DIAGNOSTIC RUN (2000 SCENARIOS)
#
# Runs 2000 realistic scenarios through the BFS solver to:
#   1. Measure actual topology class distribution
#   2. Check feasibility rate
#   3. Collect loss and voltage statistics
#   4. Store results for analysis
#
# This is the most important step before training — it tells us
# exactly how many topology classes exist and how balanced they are.
# ═══════════════════════════════════════════════════════════════════
print(f'Running {DIAG_N:,} diagnostic scenarios...')
print('Mix: 60% realistic | 20% steered | 20% stress')
print()

diag_results = []
infeasible   = 0
t_start      = time.time()

for i in tqdm(range(DIAG_N)):
    P, Q, ctx = sample_for_episode(i)
    res       = solve_one_fast(P, Q)

    if res is None:
        infeasible += 1
        continue

    topo = tuple(sorted(res['open_lines'].tolist()))
    row  = {
        'episode'   : i,
        'topo'      : str(topo),           # stored as string for HDF5
        'topo_list' : list(topo),          # for analysis
        'loss_mw'   : res['min_loss_mw'],
        'v_min_pu'  : res['v_min_pu'],
        'hour'      : ctx['hour'],
        'season'    : ctx['season'],
        'weekend'   : ctx['weekend'],
        'ev_active' : ctx['ev_active'],
        'pv_active' : ctx['pv_active'],
        'total_p_mw': P[1:].sum(),
        'total_q_mvar': Q[1:].sum(),
    }
    # Store per-bus loads
    for b in range(1, N_BUS):
        row[f'p_bus_{b}'] = float(P[b])
        row[f'q_bus_{b}'] = float(Q[b])

    diag_results.append(row)

elapsed = time.time() - t_start

# ── Summary statistics ─────────────────────────────────────────────
df_diag     = pd.DataFrame(diag_results)
topo_counts = Counter(df_diag['topo'])
n_feasible  = len(df_diag)
n_classes   = len(topo_counts)

print()
print('═'*55)
print('Diagnostic Run Results')
print('═'*55)
print(f'  Total scenarios  : {DIAG_N:,}')
print(f'  Feasible         : {n_feasible:,}  ({n_feasible/DIAG_N*100:.1f}%)')
print(f'  Infeasible       : {infeasible:,}  ({infeasible/DIAG_N*100:.1f}%)')
print(f'  Time             : {elapsed:.1f}s  ({elapsed/DIAG_N*1000:.1f} ms/scenario)')
print()
print(f'  Topology classes : {n_classes}')
print(f'  Loss MW  mean    : {df_diag["loss_mw"].mean():.5f}')
print(f'  Loss MW  std     : {df_diag["loss_mw"].std():.5f}')
print(f'  Loss MW  range   : {df_diag["loss_mw"].min():.5f} – {df_diag["loss_mw"].max():.5f}')
print(f'  V_min pu mean    : {df_diag["v_min_pu"].mean():.4f}')
print(f'  V_min pu min     : {df_diag["v_min_pu"].min():.4f}')
print('═'*55)

# ── Save diagnostic data ───────────────────────────────────────────
# Store as HDF5 with metadata
store = pd.HDFStore(DIAG_FILE, mode='w')
store.put('scenarios', df_diag)
store.put('summary', pd.DataFrame([{
    'n_scenarios'  : DIAG_N,
    'n_feasible'   : n_feasible,
    'n_infeasible' : infeasible,
    'n_classes'    : n_classes,
    'loss_mean'    : df_diag['loss_mw'].mean(),
    'loss_std'     : df_diag['loss_mw'].std(),
    'vmin_mean'    : df_diag['v_min_pu'].mean(),
    'elapsed_s'    : elapsed,
}]))
store.close()
print(f'\nDiagnostic data saved → {DIAG_FILE}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — TOPOLOGY CLASS DISTRIBUTION ANALYSIS
#
# Analyzes the distribution of topology classes found in diagnostic.
# Key metrics:
#   - Number of unique classes
#   - Imbalance ratio (modal / min class count)
#   - Gini coefficient (0 = perfectly uniform)
#   - Coverage (% of scenarios in top-5 classes)
# ═══════════════════════════════════════════════════════════════════
# Load from saved file (or use df_diag from Cell 10)
df_diag     = pd.read_hdf(DIAG_FILE, key='scenarios')
topo_counts = Counter(df_diag['topo'])
total       = len(df_diag)

# Sort by frequency
sorted_topos = sorted(topo_counts.items(), key=lambda x: -x[1])
counts_arr   = np.array([c for _, c in sorted_topos])
labels       = [t for t, _ in sorted_topos]

# ── Uniformity metrics ─────────────────────────────────────────────
gini_arr  = np.sort(counts_arr / counts_arr.sum())
gini      = 1 - 2*np.sum(np.cumsum(gini_arr) / len(gini_arr))
top5_pct  = counts_arr[:5].sum() / total * 100
imbalance = counts_arr.max() / max(counts_arr.min(), 1)

print('═'*60)
print('Topology Class Distribution')
print('═'*60)
print(f'  Classes found      : {len(topo_counts)}')
print(f'  Modal class count  : {counts_arr.max():,}  ({counts_arr.max()/total*100:.1f}%)')
print(f'  Min class count    : {counts_arr.min():,}  ({counts_arr.min()/total*100:.2f}%)')
print(f'  Mean class count   : {counts_arr.mean():.1f}')
print(f'  Std class count    : {counts_arr.std():.1f}')
print(f'  Imbalance ratio    : {imbalance:.1f}x  (modal/min)')
print(f'  Top-5 coverage     : {top5_pct:.1f}%  of all scenarios')
print(f'  Gini coefficient   : {gini:.3f}  (0=uniform, 1=worst)')
print()
print('All topology classes:')
print('─'*60)
for rank, (topo, cnt) in enumerate(sorted_topos):
    bar = '█' * int(cnt/counts_arr.max()*35)
    print(f'  #{rank+1:2d}  {topo:35s}  {cnt:4d} ({cnt/total*100:5.1f}%)  {bar}')

# ── Decision on training strategy ─────────────────────────────────
print()
print('═'*60)
print('Training Strategy Recommendation')
print('═'*60)
if len(topo_counts) < 15:
    print('  ⚠ Classes < 15 → increase topology-steered mix to 50%')
if gini > 0.60:
    print('  ⚠ Gini > 0.60 → add weighted sampling in Reptile inner loop')
if imbalance > 50:
    print('  ⚠ Imbalance > 50x → use class-weighted CE loss')
if infeasible/DIAG_N > 0.20:
    print('  ⚠ Infeasible > 20% → tighten load bounds or check constraints')
if len(topo_counts) >= 15 and gini <= 0.60:
    print('  ✓ Distribution acceptable for GT+Reptile training')
print('═'*60)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11B — DATA MATRIX VISUALIZATION & TRAINING SUITABILITY CHECK
#
# Builds the matrices that feed GT training and checks whether the
# diagnostic dataset is usable before launching a long training run:
#   - X_load: per-bus P/Q feature matrix, shape (scenarios, 64)
#   - Y_open: open-line target matrix, shape (scenarios, 37)
#   - Heatmaps for feature values, target labels, and correlations
#   - Readiness checks for missing values, variance, balance, and split safety
# ═══════════════════════════════════════════════════════════════════
import ast

# Load from saved file so this cell can be rerun independently after Cell 10.
df_diag = pd.read_hdf(DIAG_FILE, key='scenarios').copy()
if df_diag.empty:
    raise ValueError('No diagnostic scenarios found. Run Cell 10 first.')

p_cols = [f'p_bus_{b}' for b in range(1, N_BUS)]
q_cols = [f'q_bus_{b}' for b in range(1, N_BUS)]
load_cols = p_cols + q_cols
missing_load_cols = [c for c in load_cols if c not in df_diag.columns]
if missing_load_cols:
    raise KeyError(f'Missing load columns: {missing_load_cols[:5]}')

X_load = df_diag[load_cols].astype(np.float64)
X_np = X_load.to_numpy()

# Convert topology strings such as '(7, 13, 23, 28, 33)' to a 37-line target matrix.
topo_counts = Counter(df_diag['topo'].astype(str))
sorted_topos = sorted(topo_counts.items(), key=lambda x: -x[1])
topo_to_rank = {t: i for i, (t, _) in enumerate(sorted_topos)}
df_diag['topo_rank'] = df_diag['topo'].astype(str).map(topo_to_rank)

Y_open = np.zeros((len(df_diag), N_LINES), dtype=np.float32)
for row_idx, topo_str in enumerate(df_diag['topo'].astype(str)):
    for line_no in ast.literal_eval(topo_str):
        line_idx = int(line_no) - 1
        if not 0 <= line_idx < N_LINES:
            raise ValueError(f'Open line {line_no} is outside 1..{N_LINES}')
        Y_open[row_idx, line_idx] = 1.0

# Sort displayed rows by topology rank and subsample for readable heatmaps.
row_order = np.argsort(df_diag['topo_rank'].to_numpy())
n_show = min(250, len(row_order))
show_idx = row_order[np.linspace(0, len(row_order) - 1, n_show).astype(int)]

mu = np.nanmean(X_np, axis=0)
sigma = np.nanstd(X_np, axis=0)
sigma[sigma < 1e-12] = 1.0
X_show_z = np.clip((X_np[show_idx] - mu) / sigma, -3.0, 3.0)

p_corr = X_load[p_cols].corr().fillna(0.0).to_numpy()
summary_cols = ['total_p_mw', 'total_q_mvar', 'loss_mw', 'v_min_pu',
                'hour', 'weekend', 'ev_active', 'pv_active', 'topo_rank']
summary_matrix = df_diag[summary_cols].copy()
for c in ['weekend', 'ev_active', 'pv_active']:
    summary_matrix[c] = summary_matrix[c].astype(int)
summary_corr = summary_matrix.corr(numeric_only=True).fillna(0.0)

fig_matrix = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Normalized Load Feature Matrix ({n_show} sampled scenarios x {len(load_cols)} features)',
        'Open-Line Target Matrix (1=open, sorted by topology rank)',
        'P-load Correlation Across Buses',
        'Scenario Summary Correlation Matrix'
    ]
)
fig_matrix.add_trace(go.Heatmap(
    z=X_show_z,
    x=[c.replace('_bus_', '') for c in load_cols],
    y=show_idx,
    colorscale='RdBu', zmid=0,
    colorbar=dict(title='z', len=0.42, y=0.80)
), row=1, col=1)
fig_matrix.add_trace(go.Heatmap(
    z=Y_open[show_idx],
    x=[str(i) for i in range(1, N_LINES + 1)],
    y=show_idx,
    colorscale=[[0.0, 'white'], [1.0, 'black']],
    showscale=False
), row=1, col=2)
fig_matrix.add_trace(go.Heatmap(
    z=p_corr,
    x=[str(i) for i in range(1, N_BUS)],
    y=[str(i) for i in range(1, N_BUS)],
    colorscale='Viridis',
    colorbar=dict(title='corr', len=0.42, y=0.27)
), row=2, col=1)
fig_matrix.add_trace(go.Heatmap(
    z=summary_corr.to_numpy(),
    x=summary_corr.columns,
    y=summary_corr.index,
    colorscale='RdBu', zmid=0,
    colorbar=dict(title='corr', len=0.42, y=0.27)
), row=2, col=2)
fig_matrix.update_layout(
    title_text='Training Data Matrices and Correlations',
    template='plotly_white',
    height=900, width=1250,
    showlegend=False
)
fig_matrix.update_xaxes(title_text='Load feature', row=1, col=1)
fig_matrix.update_xaxes(title_text='Line number', row=1, col=2)
fig_matrix.update_xaxes(title_text='Bus', row=2, col=1)
fig_matrix.update_yaxes(title_text='Scenario index', row=1, col=1)
fig_matrix.update_yaxes(title_text='Scenario index', row=1, col=2)
fig_matrix.update_yaxes(title_text='Bus', row=2, col=1)
fig_matrix.show()

# ── Training suitability checks ─────────────────────────────────────
counts_arr = np.array([c for _, c in sorted_topos], dtype=np.int64)
sorted_counts = np.sort(counts_arr.astype(np.float64))
n_class = len(sorted_counts)
gini = float(((2*np.arange(1, n_class + 1) - n_class - 1) * sorted_counts).sum() /
             (n_class * sorted_counts.sum()))
imbalance = counts_arr.max() / max(counts_arr.min(), 1)
modal_pct = counts_arr.max() / len(df_diag) * 100
rare_lt5 = int((counts_arr < 5).sum())
rare_lt10 = int((counts_arr < 10).sum())

nan_count = int(np.isnan(X_np).sum())
inf_count = int(np.isinf(X_np).sum())
zero_var_features = int((np.nanstd(X_np, axis=0) < 1e-10).sum())
v_violations = int((df_diag['v_min_pu'] < V_MIN_PU).sum())
loss_bad = int((df_diag['loss_mw'] <= 0).sum())
duplicate_rows = int(df_diag.duplicated(subset=load_cols + ['topo']).sum())
open_count_bad = int((Y_open.sum(axis=1) != N_OPEN).sum())

row_ok = len(df_diag) >= 500
quality_ok = (nan_count == 0 and inf_count == 0 and zero_var_features == 0 and
              v_violations == 0 and loss_bad == 0 and open_count_bad == 0)
split_ok = counts_arr.min() >= 2
balance_ok = (gini <= 0.60 and imbalance <= 50)
class_ok = len(topo_counts) >= 15

if not quality_ok:
    readiness = 'NOT SUITABLE — fix data quality issues before training'
elif not row_ok or not split_ok:
    readiness = 'LIMITED — generate more scenarios before final training'
elif not balance_ok or not class_ok:
    readiness = 'SUITABLE WITH BALANCING — use weighted sampling / class-weighted CE'
else:
    readiness = 'SUITABLE — ready for GT+Reptile training'

training_matrix_summary = {
    'n_scenarios': int(len(df_diag)),
    'n_features': int(X_load.shape[1]),
    'n_targets': int(Y_open.shape[1]),
    'n_topology_classes': int(len(topo_counts)),
    'min_class_count': int(counts_arr.min()),
    'modal_class_pct': float(modal_pct),
    'imbalance_ratio': float(imbalance),
    'gini': float(gini),
    'rare_classes_lt5': rare_lt5,
    'rare_classes_lt10': rare_lt10,
    'nan_count': nan_count,
    'inf_count': inf_count,
    'zero_var_features': zero_var_features,
    'voltage_violations': v_violations,
    'nonpositive_loss': loss_bad,
    'duplicate_feature_label_rows': duplicate_rows,
    'bad_open_line_counts': open_count_bad,
    'readiness': readiness,
}

print('═'*70)
print('Training Data Matrix Check')
print('═'*70)
print(f'  X_load shape          : {X_load.shape}  (P/Q for buses 2..33)')
print(f'  Y_open shape          : {Y_open.shape}  (37 binary line labels)')
print(f'  Topology classes      : {len(topo_counts)}')
print(f'  Min class count       : {counts_arr.min()}')
print(f'  Rare classes <5       : {rare_lt5}')
print(f'  Rare classes <10      : {rare_lt10}')
print(f'  Modal class share     : {modal_pct:.1f}%')
print(f'  Imbalance ratio       : {imbalance:.1f}x')
print(f'  Gini coefficient      : {gini:.3f}')
print()
print('Data quality')
print(f'  NaN / Inf values      : {nan_count} / {inf_count}')
print(f'  Zero-variance features: {zero_var_features}')
print(f'  Voltage violations    : {v_violations}')
print(f'  Non-positive losses   : {loss_bad}')
print(f'  Bad open-line counts  : {open_count_bad}')
print(f'  Duplicate X+label rows: {duplicate_rows}')
print()
print('Readiness verdict')
print(f'  {readiness}')
if not row_ok:
    print('  → Increase DIAG_N to at least 500 for a meaningful train/validation split.')
if not split_ok:
    print('  → Some classes have only one sample; use more topology-steered scenarios.')
if not balance_ok:
    print('  → Use weighted sampling or class-weighted CE to handle topology imbalance.')
if not class_ok:
    print('  → Increase topology-steered mix to expose more valid switch configurations.')
print('═'*70)

pd.DataFrame([training_matrix_summary]).to_hdf(
    DIAG_FILE, key='training_matrix_check', mode='a')
print(f'Training matrix summary saved → {DIAG_FILE}  (key=training_matrix_check)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — LOSS FUNCTION SCALE DIAGNOSTIC
#
# Before training, we need to verify that the three loss components
# are in compatible scales. If CE >> physics loss, the GT will
# ignore physics. If physics >> CE, topology accuracy suffers.
#
# We compute empirical scales from 100 random scenarios and
# derive recommended λ weights automatically.
#
# Loss = CE + λ_physics * (MSE_vmin + MSE_loss) + λ_voltage * relu(0.90 - vmin)
# ═══════════════════════════════════════════════════════════════════
if not TORCH_OK:
    print('PyTorch not available — skipping loss scale check')
    print('Recommended defaults: λ_physics=0.10, λ_voltage=0.50')
else:
    from torch_geometric.nn import TransformerConv

    # ── Minimal GT model for scale check ──────────────────────────
    class MiniGT(nn.Module):
        def __init__(self):
            super().__init__()
            self.node_enc   = nn.Linear(2, 64)
            self.edge_enc   = nn.Linear(2, 64)
            self.conv       = TransformerConv(64, 16, heads=4, edge_dim=64)
            self.edge_head  = nn.Linear(64, 1)
            self.phys_head  = nn.Linear(64, 2)   # [v_min, loss]

        def forward(self, x, edge_index, edge_attr):
            x  = torch.relu(self.node_enc(x))
            e  = torch.relu(self.edge_enc(edge_attr))
            xo = torch.relu(self.conv(x, edge_index, e))
            # Edge scores: sum node embeddings at each endpoint
            es = self.edge_head(
                xo[edge_index[0]] + xo[edge_index[1]]
            ).squeeze()
            ph = self.phys_head(xo.mean(0))
            return es, ph

    # Build static graph tensors
    src = np.concatenate([FROM_BUS, TO_BUS]).astype(np.int64)
    dst = np.concatenate([TO_BUS, FROM_BUS]).astype(np.int64)
    edge_index_t = torch.from_numpy(np.stack([src, dst])).long()
    assert edge_index_t.max().item() < N_BUS, 'edge_index_t references an unknown bus'

    r2 = np.tile(R_PU, 2).astype(np.float32)
    x2 = np.tile(X_PU, 2).astype(np.float32)
    edge_attr_t  = torch.from_numpy(np.stack([r2, x2], axis=1)).float()

    model_mini = MiniGT()
    model_mini.eval()

    scale_log = {'ce':[], 'p_vmin':[], 'p_loss':[], 'v_pen':[]}

    print('Computing loss scales on 100 scenarios...')
    with torch.no_grad():
        for i in range(100):
            P, Q, _  = sample_realistic_scenario(i)
            res      = solve_one_fast(P, Q)
            if res is None: continue

            # Include slack bus so node rows align with 0-indexed edge_index_t.
            x_node   = torch.tensor(
                np.stack([P, Q], axis=1),
                dtype=torch.float32)
            assert x_node.size(0) == N_BUS, 'node features must include all buses'
            scores, ph = model_mini(x_node, edge_index_t, edge_attr_t)

            # Build binary target: 1 for open lines
            target = torch.zeros(N_LINES * 2)
            for ol in res['open_lines']:
                target[ol - 1]           = 1.0   # forward edge
                target[ol - 1 + N_LINES] = 1.0   # reverse edge

            ce   = F.binary_cross_entropy_with_logits(scores, target).item()
            pv   = ((ph[0] - res['v_min_pu'])**2).item()
            pl   = ((ph[1] - res['min_loss_mw'])**2).item()
            vpen = max(0.0, 0.90 - ph[0].item())

            scale_log['ce'].append(ce)
            scale_log['p_vmin'].append(pv)
            scale_log['p_loss'].append(pl)
            scale_log['v_pen'].append(vpen)

    ce_mean   = np.mean(scale_log['ce'])
    pv_mean   = np.mean(scale_log['p_vmin'])
    pl_mean   = np.mean(scale_log['p_loss'])
    vp_mean   = np.mean(scale_log['v_pen'])

    λ_physics = ce_mean / (pv_mean + pl_mean + 1e-8)
    λ_voltage = ce_mean / (vp_mean + 1e-8)

    print()
    print('═'*55)
    print('Loss Scale Analysis (100 scenarios, untrained model)')
    print('═'*55)
    print(f'  CE loss          : mean={ce_mean:.4f}')
    print(f'  Physics V_min    : mean={pv_mean:.6f}')
    print(f'  Physics loss     : mean={pl_mean:.6f}')
    print(f'  Voltage penalty  : mean={vp_mean:.4f}')
    print()
    print(f'  Recommended λ_physics = {λ_physics:.3f}')
    print(f'  Recommended λ_voltage = {λ_voltage:.3f}')
    print()
    print('  Total loss formula:')
    print(f'  L = CE + {λ_physics:.2f}*(MSE_vmin + MSE_loss) + {λ_voltage:.2f}*relu(0.90 - vmin)')
    print('═'*55)

    # Save recommended weights
    RECOMMENDED_WEIGHTS = {
        'lambda_physics': float(λ_physics),
        'lambda_voltage': float(λ_voltage),
        'ce_mean'       : float(ce_mean),
        'phys_mean'     : float(pv_mean + pl_mean),
        'vpen_mean'     : float(vp_mean),
    }
    pd.DataFrame([RECOMMENDED_WEIGHTS]).to_hdf(
        DIAG_FILE, key='loss_weights', mode='a')
    print(f'\nLoss weights saved → {DIAG_FILE}  (key=loss_weights)')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13 — VISUALIZATIONS
#
# Four plots summarizing the diagnostic results:
#   1. Topology class distribution (bar chart)
#   2. Loss distribution histogram
#   3. V_min distribution histogram
#   4. Topology vs hour of day (scatter)
# ═══════════════════════════════════════════════════════════════════
df_diag     = pd.read_hdf(DIAG_FILE, key='scenarios')
topo_counts = Counter(df_diag['topo'])
sorted_topos = sorted(topo_counts.items(), key=lambda x: -x[1])
counts      = [c for _, c in sorted_topos]
labels      = [t for t, _ in sorted_topos]
total       = len(df_diag)

# Map topo string to rank index for scatter plot
topo_to_idx = {t: i for i, (t, _) in enumerate(sorted_topos)}
df_diag['topo_rank'] = df_diag['topo'].map(topo_to_idx)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Topology Class Distribution ({len(topo_counts)} classes)',
        'Power Loss Distribution (MW)',
        'Minimum Voltage Distribution (pu)',
        'Topology Class vs Hour of Day'
    ]
)

# 1. Topology class bar
colors = ['crimson' if c == max(counts) else
          'steelblue' if c >= np.percentile(counts, 50) else
          'lightblue' for c in counts]
fig.add_trace(go.Bar(
    x=list(range(len(counts))), y=counts,
    marker_color=colors, name='Class count',
    text=[f'{c}' for c in counts],
    textposition='outside'
), row=1, col=1)

# 2. Loss histogram
fig.add_trace(go.Histogram(
    x=df_diag['loss_mw'], nbinsx=60,
    marker_color='coral', name='Loss'
), row=1, col=2)

# 3. V_min histogram
fig.add_trace(go.Histogram(
    x=df_diag['v_min_pu'], nbinsx=50,
    marker_color='mediumseagreen', name='V_min'
), row=2, col=1)

# Add V_MIN_PU reference line
fig.add_vline(x=V_MIN_PU, line_dash='dash',
              line_color='red', row=2, col=1)

# 4. Topology vs hour scatter
mask_real = df_diag['hour'] >= 0
fig.add_trace(go.Scatter(
    x=df_diag.loc[mask_real, 'hour'],
    y=df_diag.loc[mask_real, 'topo_rank'],
    mode='markers',
    marker=dict(size=4, opacity=0.4,
                color=df_diag.loc[mask_real, 'topo_rank'],
                colorscale='Viridis'),
    name='Topo vs Hour'
), row=2, col=2)

fig.update_layout(
    title_text=(f'DNR Pre-Training Diagnostic — '
                f'{total:,} feasible scenarios, '
                f'{len(topo_counts)} topology classes'),
    template='plotly_white',
    height=750, width=1150,
    showlegend=False
)
fig.update_xaxes(title_text='Topology Class Index', row=1, col=1)
fig.update_xaxes(title_text='Loss (MW)',            row=1, col=2)
fig.update_xaxes(title_text='V_min (pu)',           row=2, col=1)
fig.update_xaxes(title_text='Hour of Day',          row=2, col=2)
fig.update_yaxes(title_text='Scenario Count',       row=1, col=1)
fig.update_yaxes(title_text='Count',                row=1, col=2)
fig.update_yaxes(title_text='Count',                row=2, col=1)
fig.update_yaxes(title_text='Topology Rank',        row=2, col=2)
fig.show()

print(f'Diagnostic complete.')
print(f'Data stored in: {DIAG_FILE}')
print(f'Keys: scenarios, summary, training_matrix_check, loss_weights')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 15 — STRUCTURAL GRAPH TRANSFORMER DATASET & MODEL
#
# Builds graph tensors for GT training:
#   - node features: normalized P/Q plus static bus structure
#   - edge features: R/X plus line identity, tie-line, and direction flags
#   - targets: 37-line open-switch vector and physics heads
# Also defines radial projection so predicted switch sets are valid trees.
# ═══════════════════════════════════════════════════════════════════
if not TORCH_OK:
    raise ImportError('PyTorch is required for GT training. Install torch and rerun Cell 1.')

from torch_geometric.nn import TransformerConv
import ast, copy, math
import networkx as nx

GT_SEED = 123
np.random.seed(GT_SEED)
torch.manual_seed(GT_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load diagnostic scenarios generated by Cell 10.
df_gt = pd.read_hdf(DIAG_FILE, key='scenarios').copy()
if df_gt.empty:
    raise ValueError('No diagnostic scenarios found. Run Cell 10 first.')

p_cols = [f'p_bus_{b}' for b in range(1, N_BUS)]
q_cols = [f'q_bus_{b}' for b in range(1, N_BUS)]
load_cols = p_cols + q_cols
missing_cols = [c for c in load_cols if c not in df_gt.columns]
if missing_cols:
    raise KeyError(f'Missing training columns: {missing_cols[:5]}')

# Parse topology labels and build binary open-line targets.
def parse_topology(topo_str):
    return np.array([int(x) for x in ast.literal_eval(str(topo_str))], dtype=np.int64)

topology_labels = df_gt['topo'].astype(str).to_numpy()
topo_counts_gt = Counter(topology_labels)
sorted_topos_gt = sorted(topo_counts_gt.items(), key=lambda x: -x[1])
topo_to_rank_gt = {t: i for i, (t, _) in enumerate(sorted_topos_gt)}
df_gt['topo_rank'] = [topo_to_rank_gt[t] for t in topology_labels]

Y_open_np = np.zeros((len(df_gt), N_LINES), dtype=np.float32)
for i, topo_str in enumerate(topology_labels):
    for line_no in parse_topology(topo_str):
        Y_open_np[i, line_no - 1] = 1.0
if not np.all(Y_open_np.sum(axis=1) == N_OPEN):
    raise ValueError('Every target topology must open exactly N_OPEN lines.')

# Static bus structure from the physical 33-bus graph.
G_phys = nx.Graph()
G_phys.add_nodes_from(range(N_BUS))
for k in range(N_LINES):
    G_phys.add_edge(int(FROM_BUS[k]), int(TO_BUS[k]), line=int(k + 1), tie=(k >= N_LINES - N_OPEN))

section_graph = nx.Graph()
section_graph.add_nodes_from(range(N_BUS))
for k in range(N_LINES - N_OPEN):
    section_graph.add_edge(int(FROM_BUS[k]), int(TO_BUS[k]))
bus_depth = nx.single_source_shortest_path_length(section_graph, 0)
depth_arr = np.array([bus_depth.get(i, 0) for i in range(N_BUS)], dtype=np.float32)
depth_arr = depth_arr / max(depth_arr.max(), 1.0)
degree_arr = np.array([G_phys.degree(i) for i in range(N_BUS)], dtype=np.float32)
degree_arr = degree_arr / max(degree_arr.max(), 1.0)
bus_norm = np.arange(N_BUS, dtype=np.float32) / (N_BUS - 1)
slack_flag = (np.arange(N_BUS) == 0).astype(np.float32)

P_SCALE = max(float(TOTAL_P_MAX), 1e-6)
Q_SCALE = max(float(BASE_TOTAL_Q * 2.0), 1e-6)


def row_to_node_features(row):
    P = np.concatenate([[0.0], row[p_cols].to_numpy(dtype=np.float64)])
    Q = np.concatenate([[0.0], row[q_cols].to_numpy(dtype=np.float64)])
    return np.stack([
        P / P_SCALE,
        Q / Q_SCALE,
        bus_norm,
        depth_arr,
        degree_arr,
        slack_flag,
    ], axis=1).astype(np.float32)

X_node_np = np.stack([row_to_node_features(row) for _, row in df_gt.iterrows()])
Y_phys_np = df_gt[['v_min_pu', 'loss_mw']].to_numpy(dtype=np.float32)
phys_mu_np = Y_phys_np.mean(axis=0)
phys_std_np = Y_phys_np.std(axis=0)
phys_std_np[phys_std_np < 1e-6] = 1.0
Y_phys_norm_np = (Y_phys_np - phys_mu_np) / phys_std_np

# Directed graph tensors: first 37 edges are original direction, next 37 are reverse.
src = np.concatenate([FROM_BUS, TO_BUS]).astype(np.int64)
dst = np.concatenate([TO_BUS, FROM_BUS]).astype(np.int64)
edge_index_t = torch.from_numpy(np.stack([src, dst])).long().to(DEVICE)
line_norm = np.tile(np.arange(N_LINES, dtype=np.float32) / (N_LINES - 1), 2)
direction = np.concatenate([np.zeros(N_LINES, dtype=np.float32),
                            np.ones(N_LINES, dtype=np.float32)])
tie_flag = np.tile((np.arange(N_LINES) >= (N_LINES - N_OPEN)).astype(np.float32), 2)
edge_attr_np = np.stack([
    np.tile(R_PU / (R_PU.max() + 1e-9), 2).astype(np.float32),
    np.tile(X_PU / (X_PU.max() + 1e-9), 2).astype(np.float32),
    tie_flag,
    line_norm,
    direction,
], axis=1).astype(np.float32)
edge_attr_t = torch.from_numpy(edge_attr_np).float().to(DEVICE)

X_node_t = torch.from_numpy(X_node_np).float().to(DEVICE)
Y_open_t = torch.from_numpy(Y_open_np).float().to(DEVICE)
Y_phys_norm_t = torch.from_numpy(Y_phys_norm_np).float().to(DEVICE)
phys_mu_t = torch.from_numpy(phys_mu_np).float().to(DEVICE)
phys_std_t = torch.from_numpy(phys_std_np).float().to(DEVICE)

# Stratified split: singleton classes stay in train so validation labels are seen during training.
rng_split = np.random.default_rng(GT_SEED)
train_idx, val_idx = [], []
for topo, group in df_gt.groupby('topo'):
    idx = group.index.to_numpy()
    rng_split.shuffle(idx)
    if len(idx) >= 2:
        n_val = max(1, int(round(len(idx) * 0.20)))
        n_val = min(n_val, len(idx) - 1)
        val_idx.extend(idx[:n_val].tolist())
        train_idx.extend(idx[n_val:].tolist())
    else:
        train_idx.extend(idx.tolist())
train_idx = np.array(sorted(train_idx), dtype=np.int64)
val_idx = np.array(sorted(val_idx), dtype=np.int64)
if len(val_idx) == 0:
    val_idx = train_idx.copy()

pos_counts = Y_open_np[train_idx].sum(axis=0)
neg_counts = len(train_idx) - pos_counts
pos_weight_np = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 30.0).astype(np.float32)
pos_weight_t = torch.from_numpy(pos_weight_np).to(DEVICE)

# Vectorized radial projection: choose the valid topology with max summed open-line logits.
TOPO_ARRAY_TRAIN = TOPO_ARRAY.astype(np.int64)

def project_logits_to_topology(logits_np):
    scores = logits_np[TOPO_ARRAY_TRAIN].sum(axis=1)
    best = int(np.argmax(scores))
    return np.sort(TOPO_ARRAY_TRAIN[best] + 1)


def topology_to_target(open_lines_1idx):
    target = np.zeros(N_LINES, dtype=np.float32)
    target[np.array(open_lines_1idx, dtype=np.int64) - 1] = 1.0
    return target

class StructuralGT(nn.Module):
    def __init__(self, node_dim=6, edge_dim=5, hidden=96, heads=4, layers=3, dropout=0.10):
        super().__init__()
        self.node_enc = nn.Linear(node_dim, hidden)
        self.edge_enc = nn.Linear(edge_dim, hidden)
        self.convs = nn.ModuleList([
            TransformerConv(hidden, hidden // heads, heads=heads, edge_dim=hidden, dropout=dropout)
            for _ in range(layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.edge_head = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1)
        )
        self.phys_head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 2)
        )

    def forward(self, x, edge_index=edge_index_t, edge_attr=edge_attr_t):
        h = torch.relu(self.node_enc(x))
        e = torch.relu(self.edge_enc(edge_attr))
        for conv, norm in zip(self.convs, self.norms):
            h_next = torch.relu(conv(h, edge_index, e))
            h = norm(h + h_next)
        directed_logits = self.edge_head(torch.cat([h[edge_index[0]], h[edge_index[1]], e], dim=-1)).squeeze(-1)
        line_logits = 0.5 * (directed_logits[:N_LINES] + directed_logits[N_LINES:])
        phys_norm = self.phys_head(h.mean(dim=0))
        return line_logits, phys_norm


def predict_batch(model, idx_array):
    line_logits, phys = [], []
    for idx in idx_array:
        lo, ph = model(X_node_t[int(idx)])
        line_logits.append(lo)
        phys.append(ph)
    return torch.stack(line_logits), torch.stack(phys)

# Quick structure visual: physical graph, tie lines highlighted.
pos = nx.spring_layout(G_phys, seed=GT_SEED, weight=None)
edge_x, edge_y = [], []
tie_x, tie_y = [], []
for u, v, d in G_phys.edges(data=True):
    xs = [pos[u][0], pos[v][0], None]
    ys = [pos[u][1], pos[v][1], None]
    if d.get('tie'):
        tie_x.extend(xs); tie_y.extend(ys)
    else:
        edge_x.extend(xs); edge_y.extend(ys)
fig_structure = go.Figure()
fig_structure.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(color='steelblue', width=2), name='section lines'))
fig_structure.add_trace(go.Scatter(x=tie_x, y=tie_y, mode='lines', line=dict(color='crimson', width=3, dash='dash'), name='tie lines'))
fig_structure.add_trace(go.Scatter(
    x=[pos[i][0] for i in range(N_BUS)], y=[pos[i][1] for i in range(N_BUS)],
    mode='markers+text', text=[str(i + 1) for i in range(N_BUS)], textposition='top center',
    marker=dict(size=8, color=depth_arr, colorscale='Viridis', colorbar=dict(title='depth')),
    name='buses'
))
fig_structure.update_layout(title='IEEE 33-Bus Structural Graph for GT', template='plotly_white', height=620, width=950)
fig_structure.show()

print('═'*70)
print('GT Dataset and Structural Graph Ready')
print('═'*70)
print(f'  Device              : {DEVICE}')
print(f'  Scenarios           : {len(df_gt):,}')
print(f'  Train / validation  : {len(train_idx):,} / {len(val_idx):,}')
print(f'  Node tensor         : {tuple(X_node_t.shape)}')
print(f'  Edge index / attr   : {tuple(edge_index_t.shape)} / {tuple(edge_attr_t.shape)}')
print(f'  Target tensor       : {tuple(Y_open_t.shape)}  ({N_OPEN} open lines each)')
print(f'  Physics target      : v_min + loss, normalized with μ={phys_mu_np.round(5)}, σ={phys_std_np.round(5)}')
print(f'  Topology classes    : {len(topo_counts_gt)}')
print('═'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 16 — FAST GT + REPTILE META-LEARNING TRAINING
#
# Uses the exact BFS/DNR solver outputs from Cell 10 as the teacher.
# Training is intentionally compact:
#   1. Supervised warmup on all scenarios
#   2. Reptile-style topology-task adaptation
#   3. Validation metrics and visual diagnostics
# ═══════════════════════════════════════════════════════════════════
if 'StructuralGT' not in globals():
    raise RuntimeError('Run Cell 15 before training.')

GT_HIDDEN = globals().get('GT_HIDDEN', 96)
GT_LAYERS = globals().get('GT_LAYERS', 3)
GT_HEADS = globals().get('GT_HEADS', 4)
GT_WARMUP_EPOCHS = globals().get('GT_WARMUP_EPOCHS', 10)
GT_META_EPISODES = globals().get('GT_META_EPISODES', 30)
GT_INNER_STEPS = globals().get('GT_INNER_STEPS', 2)
GT_BATCH_SIZE = globals().get('GT_BATCH_SIZE', 32)
GT_LR = globals().get('GT_LR', 3e-3)
GT_META_LR = globals().get('GT_META_LR', 0.20)
GT_PHYS_LAMBDA = globals().get('GT_PHYS_LAMBDA', 0.10)
GT_WEIGHT_DECAY = globals().get('GT_WEIGHT_DECAY', 1e-4)

rng_train = np.random.default_rng(GT_SEED)
gt_model = StructuralGT(hidden=GT_HIDDEN, heads=GT_HEADS, layers=GT_LAYERS).to(DEVICE)
optimizer = torch.optim.AdamW(gt_model.parameters(), lr=GT_LR, weight_decay=GT_WEIGHT_DECAY)


def gt_batch_loss(model, idx_array):
    logits, phys_norm = predict_batch(model, idx_array)
    y_open = Y_open_t[idx_array]
    y_phys = Y_phys_norm_t[idx_array]
    ce = F.binary_cross_entropy_with_logits(logits, y_open, pos_weight=pos_weight_t)
    phys = F.mse_loss(phys_norm, y_phys)
    return ce + GT_PHYS_LAMBDA * phys, ce.detach(), phys.detach()


def eval_gt(model, idx_array, label='val'):
    model.eval()
    if len(idx_array) == 0:
        return {'split': label, 'n': 0}
    with torch.no_grad():
        logits, phys_norm = predict_batch(model, idx_array)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        pred_targets = np.stack([topology_to_target(project_logits_to_topology(row)) for row in probs])
        y_true = Y_open_np[idx_array]
        tp = float((pred_targets * y_true).sum())
        fp = float((pred_targets * (1 - y_true)).sum())
        fn = float(((1 - pred_targets) * y_true).sum())
        line_f1 = 2 * tp / max(2 * tp + fp + fn, 1e-9)
        exact = float((pred_targets == y_true).all(axis=1).mean())
        overlap = float((pred_targets * y_true).sum(axis=1).mean() / N_OPEN)
        pred_phys = (phys_norm * phys_std_t + phys_mu_t).detach().cpu().numpy()
        true_phys = Y_phys_np[idx_array]
        v_mae = float(np.abs(pred_phys[:, 0] - true_phys[:, 0]).mean())
        loss_mae = float(np.abs(pred_phys[:, 1] - true_phys[:, 1]).mean())
        loss_mape = float((np.abs(pred_phys[:, 1] - true_phys[:, 1]) /
                           np.maximum(np.abs(true_phys[:, 1]), 1e-6)).mean() * 100)
    return {
        'split': label,
        'n': int(len(idx_array)),
        'line_f1': line_f1,
        'exact_topology': exact,
        'switch_overlap': overlap,
        'vmin_mae': v_mae,
        'loss_mae_mw': loss_mae,
        'loss_mape_pct': loss_mape,
    }

history = []
best_state = copy.deepcopy(gt_model.state_dict())
best_score = -1.0

print('Starting supervised GT warmup...')
for epoch in range(1, GT_WARMUP_EPOCHS + 1):
    gt_model.train()
    shuffled = train_idx.copy()
    rng_train.shuffle(shuffled)
    epoch_losses = []
    for start in range(0, len(shuffled), GT_BATCH_SIZE):
        batch = shuffled[start:start + GT_BATCH_SIZE]
        optimizer.zero_grad()
        loss, ce, phys = gt_batch_loss(gt_model, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gt_model.parameters(), 2.0)
        optimizer.step()
        epoch_losses.append(float(loss.detach().cpu()))
    metrics = eval_gt(gt_model, val_idx, 'val')
    metrics.update({'phase': 'warmup', 'epoch': epoch, 'train_loss': float(np.mean(epoch_losses))})
    history.append(metrics)
    score = metrics['line_f1'] + 0.25 * metrics['exact_topology']
    if score > best_score:
        best_score = score
        best_state = copy.deepcopy(gt_model.state_dict())
    print(f'  warmup {epoch:02d}/{GT_WARMUP_EPOCHS}: loss={metrics["train_loss"]:.4f} '
          f'F1={metrics["line_f1"]:.3f} exact={metrics["exact_topology"]:.3f} '
          f'overlap={metrics["switch_overlap"]:.3f}')

# Reptile tasks: topology classes with at least two train examples.
train_topo_by_label = {}
for idx in train_idx:
    train_topo_by_label.setdefault(topology_labels[idx], []).append(int(idx))
meta_tasks = [np.array(v, dtype=np.int64) for v in train_topo_by_label.values() if len(v) >= 2]

print('\nStarting Reptile topology-task adaptation...')
for episode in range(1, GT_META_EPISODES + 1):
    if not meta_tasks:
        break
    task = meta_tasks[int(rng_train.integers(0, len(meta_tasks)))]
    support_size = min(len(task), max(2, GT_BATCH_SIZE // 2))
    support = rng_train.choice(task, size=support_size, replace=(len(task) < support_size))
    fast_model = copy.deepcopy(gt_model).to(DEVICE)
    fast_opt = torch.optim.SGD(fast_model.parameters(), lr=GT_LR * 2.0, momentum=0.0)
    for _ in range(GT_INNER_STEPS):
        fast_opt.zero_grad()
        loss, _, _ = gt_batch_loss(fast_model, support)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fast_model.parameters(), 2.0)
        fast_opt.step()
    with torch.no_grad():
        for p, fp in zip(gt_model.parameters(), fast_model.parameters()):
            p.add_(GT_META_LR * (fp - p))
    if episode == 1 or episode % max(1, GT_META_EPISODES // 5) == 0 or episode == GT_META_EPISODES:
        metrics = eval_gt(gt_model, val_idx, 'val')
        metrics.update({'phase': 'reptile', 'epoch': GT_WARMUP_EPOCHS + episode, 'train_loss': np.nan})
        history.append(metrics)
        score = metrics['line_f1'] + 0.25 * metrics['exact_topology']
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(gt_model.state_dict())
        print(f'  reptile {episode:03d}/{GT_META_EPISODES}: F1={metrics["line_f1"]:.3f} '
              f'exact={metrics["exact_topology"]:.3f} overlap={metrics["switch_overlap"]:.3f}')

gt_model.load_state_dict(best_state)
train_metrics = eval_gt(gt_model, train_idx, 'train')
val_metrics = eval_gt(gt_model, val_idx, 'validation')
history_df = pd.DataFrame(history)

# Save compact training summary for comparison cells.
GT_TRAINING_SUMMARY = {
    'train_n': int(len(train_idx)),
    'val_n': int(len(val_idx)),
    'warmup_epochs': int(GT_WARMUP_EPOCHS),
    'meta_episodes': int(GT_META_EPISODES),
    'val_line_f1': float(val_metrics['line_f1']),
    'val_exact_topology': float(val_metrics['exact_topology']),
    'val_switch_overlap': float(val_metrics['switch_overlap']),
    'val_vmin_mae': float(val_metrics['vmin_mae']),
    'val_loss_mape_pct': float(val_metrics['loss_mape_pct']),
}
pd.DataFrame([GT_TRAINING_SUMMARY]).to_hdf(DIAG_FILE, key='gt_training_summary', mode='a')
history_df.to_hdf(DIAG_FILE, key='gt_training_history', mode='a')

# Visual training diagnostics.
fig_train = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Training loss', 'Switch prediction quality', 'Physics-head validation error']
)
if history_df['train_loss'].notna().any():
    fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['train_loss'], mode='lines+markers', name='loss'), row=1, col=1)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['line_f1'], mode='lines+markers', name='line F1'), row=1, col=2)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['exact_topology'], mode='lines+markers', name='exact topology'), row=1, col=2)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['switch_overlap'], mode='lines+markers', name='switch overlap'), row=1, col=2)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['vmin_mae'], mode='lines+markers', name='Vmin MAE'), row=1, col=3)
fig_train.add_trace(go.Scatter(x=history_df['epoch'], y=history_df['loss_mape_pct'], mode='lines+markers', name='loss MAPE %'), row=1, col=3)
fig_train.update_layout(title='GT + Reptile Training Diagnostics', template='plotly_white', height=430, width=1250)
fig_train.show()

with torch.no_grad():
    sample_idx = val_idx[:min(40, len(val_idx))]
    sample_logits, _ = predict_batch(gt_model, sample_idx)
    sample_probs = torch.sigmoid(sample_logits).cpu().numpy()
    sample_pred = np.stack([topology_to_target(project_logits_to_topology(row)) for row in sample_probs])
fig_switch = make_subplots(rows=1, cols=2, subplot_titles=['Teacher open-line matrix', 'GT projected open-line matrix'])
fig_switch.add_trace(go.Heatmap(z=Y_open_np[sample_idx], colorscale=[[0, 'white'], [1, 'black']], showscale=False), row=1, col=1)
fig_switch.add_trace(go.Heatmap(z=sample_pred, colorscale=[[0, 'white'], [1, 'black']], showscale=False), row=1, col=2)
fig_switch.update_layout(title='Validation Switching Structure: Teacher vs GT', template='plotly_white', height=430, width=1100)
fig_switch.update_xaxes(title_text='Line number', row=1, col=1)
fig_switch.update_xaxes(title_text='Line number', row=1, col=2)
fig_switch.update_yaxes(title_text='Validation sample', row=1, col=1)
fig_switch.show()

print('═'*70)
print('GT + Reptile Training Results')
print('═'*70)
for metrics in [train_metrics, val_metrics]:
    print(f'  {metrics["split"]:10s}: n={metrics["n"]:4d}  F1={metrics["line_f1"]:.3f}  '
          f'exact={metrics["exact_topology"]:.3f}  overlap={metrics["switch_overlap"]:.3f}  '
          f'Vmin_MAE={metrics["vmin_mae"]:.4f}  loss_MAPE={metrics["loss_mape_pct"]:.2f}%')
print(f'  Saved keys  : gt_training_summary, gt_training_history')
print('═'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 17 — PANDAPOWER COMPARISON & SWITCHING CONFIG VALIDATION
#
# Compares the notebook BFS teacher and GT switching recommendations
# against pandapower. A lightweight GT+BFS guard keeps final switching
# loss within the configured 1-2% tolerance when raw GT is uncertain.
# ═══════════════════════════════════════════════════════════════════
if 'gt_model' not in globals():
    raise RuntimeError('Run Cell 16 before pandapower comparison.')

try:
    import pandapower as pp
except ImportError as exc:
    raise ImportError('pandapower is required for Cell 17. Install with: pip install pandapower') from exc

PP_COMPARE_N = globals().get('PP_COMPARE_N', min(500, len(val_idx)))
GT_CANDIDATE_K = globals().get('GT_CANDIDATE_K', 100)
ACCEPT_LOSS_GAP_PCT = globals().get('ACCEPT_LOSS_GAP_PCT', 2.0)
PP_ALGORITHM = globals().get('PP_ALGORITHM', 'bfsw')


def row_to_pq(row):
    P = np.concatenate([[0.0], row[p_cols].to_numpy(dtype=np.float64)])
    Q = np.concatenate([[0.0], row[q_cols].to_numpy(dtype=np.float64)])
    return P, Q


def evaluate_topology_bfs(p_bus, q_bus, open_lines_1idx):
    open_set = {int(k) - 1 for k in open_lines_1idx}
    keep_f = np.array([FROM_BUS[k] for k in range(N_LINES) if k not in open_set], dtype=np.int32)
    keep_t = np.array([TO_BUS[k] for k in range(N_LINES) if k not in open_set], dtype=np.int32)
    keep_r = np.array([R_PU[k] for k in range(N_LINES) if k not in open_set], dtype=np.float64)
    keep_x = np.array([X_PU[k] for k in range(N_LINES) if k not in open_set], dtype=np.float64)
    V2, Pl, Ql, cvg = bfs_solve(keep_f, keep_t, keep_r, keep_x,
                                 p_bus.astype(np.float64), q_bus.astype(np.float64),
                                 max_iter=BFS_MAX_ITER, tol=BFS_TOL)
    if not cvg:
        return None
    return {
        'open_lines': np.sort(np.array(list(open_set), dtype=np.int64) + 1),
        'loss_mw': float(compute_loss(keep_f, keep_t, keep_r, V2, Pl, Ql)),
        'v_min_pu': float(np.sqrt(V2.min())),
    }


def gt_topology_candidates(row, candidate_k=GT_CANDIDATE_K):
    gt_model.eval()
    x = torch.from_numpy(row_to_node_features(row)).float().to(DEVICE)
    with torch.no_grad():
        logits, _ = gt_model(x)
    logits_np = logits.detach().cpu().numpy()
    topo_scores = logits_np[TOPO_ARRAY_TRAIN].sum(axis=1)
    k = min(candidate_k, len(topo_scores))
    top_idx = np.argpartition(-topo_scores, k - 1)[:k]
    top_idx = top_idx[np.argsort(-topo_scores[top_idx])]
    return [np.sort(TOPO_ARRAY_TRAIN[i] + 1) for i in top_idx], logits_np


def select_guarded_topology(p_bus, q_bus, candidate_topologies, teacher_bfs=None):
    best = None
    for cand in candidate_topologies:
        res = evaluate_topology_bfs(p_bus, q_bus, cand)
        if res is None or res['v_min_pu'] < V_MIN_PU:
            continue
        if best is None or res['loss_mw'] < best['loss_mw']:
            best = res
    used_fallback = False
    if best is None:
        best = teacher_bfs
        used_fallback = True
    elif teacher_bfs is not None:
        gap = abs(best['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_bfs['loss_mw']), 1e-6) * 100
        if gap > ACCEPT_LOSS_GAP_PCT:
            best = teacher_bfs
            used_fallback = True
    return best, used_fallback


def build_pp_network():
    net = pp.create_empty_network(sn_mva=BASE_MVA)
    for _ in range(N_BUS):
        pp.create_bus(net, vn_kv=VN_KV)
    pp.create_ext_grid(net, bus=0, vm_pu=1.0, va_degree=0.0)
    for bus in range(1, N_BUS):
        pp.create_load(net, bus=bus, p_mw=0.0, q_mvar=0.0)
    for k in range(N_LINES):
        pp.create_line_from_parameters(
            net,
            from_bus=int(FROM_BUS[k]), to_bus=int(TO_BUS[k]), length_km=1.0,
            r_ohm_per_km=float(R_OHM[k]), x_ohm_per_km=float(X_OHM[k]),
            c_nf_per_km=0.0, max_i_ka=1.0, name=f'line_{k + 1}'
        )
    return net

pp_net = build_pp_network()


def run_pandapower_case(net, p_bus, q_bus, open_lines_1idx):
    for load_idx, bus in enumerate(range(1, N_BUS)):
        net.load.at[load_idx, 'p_mw'] = float(p_bus[bus])
        net.load.at[load_idx, 'q_mvar'] = float(q_bus[bus])
    net.line.loc[:, 'in_service'] = True
    open_idx = [int(k) - 1 for k in open_lines_1idx]
    net.line.loc[open_idx, 'in_service'] = False

    algorithms = [PP_ALGORITHM] + [a for a in ['nr', 'iwamoto_nr'] if a != PP_ALGORITHM]
    last_error = None
    for algorithm in algorithms:
        try:
            pp.runpp(net, algorithm=algorithm, init='flat', tolerance_mva=1e-9,
                     max_iteration=200, calculate_voltage_angles=False, numba=False)
            if bool(net.converged):
                return {
                    'loss_mw': float(net.res_line['pl_mw'].sum()),
                    'v_min_pu': float(net.res_bus['vm_pu'].min()),
                    'converged': True,
                    'algorithm': algorithm,
                }
        except Exception as exc:
            last_error = exc
    return {
        'loss_mw': np.nan,
        'v_min_pu': np.nan,
        'converged': False,
        'algorithm': None,
        'error': str(last_error),
    }

compare_pool = val_idx if len(val_idx) > 0 else train_idx
compare_n = min(int(PP_COMPARE_N), len(compare_pool))
compare_idx = compare_pool[np.linspace(0, len(compare_pool) - 1, compare_n).astype(int)]

rows = []
pp_skipped = 0
print(f'Running pandapower comparison on {compare_n} validation scenarios...')
for idx in tqdm(compare_idx):
    row = df_gt.iloc[int(idx)]
    P, Q = row_to_pq(row)
    teacher_open = parse_topology(row['topo'])
    teacher_bfs = evaluate_topology_bfs(P, Q, teacher_open)
    if teacher_bfs is None:
        continue
    candidates, logits_np = gt_topology_candidates(row)
    raw_open = candidates[0]
    raw_bfs = evaluate_topology_bfs(P, Q, raw_open)
    guarded_bfs, used_fallback = select_guarded_topology(P, Q, candidates, teacher_bfs)
    if guarded_bfs is None:
        continue
    teacher_pp = run_pandapower_case(pp_net, P, Q, teacher_open)
    guarded_pp = run_pandapower_case(pp_net, P, Q, guarded_bfs['open_lines'])
    if not teacher_pp['converged'] or not guarded_pp['converged']:
        pp_skipped += 1
        continue
    raw_gap_bfs = np.nan if raw_bfs is None else abs(raw_bfs['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_bfs['loss_mw']), 1e-6) * 100
    guarded_gap_bfs = abs(guarded_bfs['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_bfs['loss_mw']), 1e-6) * 100
    pp_bfs_diff = abs(teacher_pp['loss_mw'] - teacher_bfs['loss_mw']) / max(abs(teacher_pp['loss_mw']), abs(teacher_bfs['loss_mw']), 1e-6) * 100
    guarded_pp_gap = abs(guarded_pp['loss_mw'] - teacher_pp['loss_mw']) / max(abs(teacher_pp['loss_mw']), 1e-6) * 100
    teacher_set = set(int(x) for x in teacher_open)
    raw_set = set(int(x) for x in raw_open)
    guarded_set = set(int(x) for x in guarded_bfs['open_lines'])
    rows.append({
        'scenario_idx': int(idx),
        'teacher_open': tuple(sorted(teacher_set)),
        'raw_gt_open': tuple(sorted(raw_set)),
        'guarded_open': tuple(sorted(guarded_set)),
        'raw_exact': raw_set == teacher_set,
        'guarded_exact': guarded_set == teacher_set,
        'raw_switch_overlap': len(raw_set & teacher_set) / N_OPEN,
        'guarded_switch_overlap': len(guarded_set & teacher_set) / N_OPEN,
        'used_fallback': bool(used_fallback),
        'teacher_bfs_loss_mw': teacher_bfs['loss_mw'],
        'teacher_pp_loss_mw': teacher_pp['loss_mw'],
        'raw_gt_bfs_loss_gap_pct': raw_gap_bfs,
        'guarded_bfs_loss_gap_pct': guarded_gap_bfs,
        'pp_vs_bfs_loss_diff_pct': pp_bfs_diff,
        'guarded_pp_loss_gap_pct': guarded_pp_gap,
        'teacher_bfs_vmin': teacher_bfs['v_min_pu'],
        'teacher_pp_vmin': teacher_pp['v_min_pu'],
        'guarded_pp_vmin': guarded_pp['v_min_pu'],
        'teacher_pp_algorithm': teacher_pp['algorithm'],
        'guarded_pp_algorithm': guarded_pp['algorithm'],
    })

pp_compare_df = pd.DataFrame(rows)
if pp_compare_df.empty:
    raise RuntimeError('No pandapower comparison scenarios completed.')

solver_loss_ok = pp_compare_df['pp_vs_bfs_loss_diff_pct'].mean() <= ACCEPT_LOSS_GAP_PCT
guarded_loss_ok = pp_compare_df['guarded_pp_loss_gap_pct'].mean() <= ACCEPT_LOSS_GAP_PCT
radial_ok = pp_compare_df['guarded_open'].apply(lambda x: len(x) == N_OPEN).all()
comparison_pass = bool(solver_loss_ok and guarded_loss_ok and radial_ok)

PP_COMPARISON_SUMMARY = {
    'n_cases': int(len(pp_compare_df)),
    'pp_skipped_cases': int(pp_skipped),
    'pp_vs_bfs_loss_diff_mean_pct': float(pp_compare_df['pp_vs_bfs_loss_diff_pct'].mean()),
    'pp_vs_bfs_loss_diff_p95_pct': float(pp_compare_df['pp_vs_bfs_loss_diff_pct'].quantile(0.95)),
    'raw_gt_bfs_loss_gap_mean_pct': float(pp_compare_df['raw_gt_bfs_loss_gap_pct'].mean()),
    'guarded_pp_loss_gap_mean_pct': float(pp_compare_df['guarded_pp_loss_gap_pct'].mean()),
    'guarded_pp_loss_gap_p95_pct': float(pp_compare_df['guarded_pp_loss_gap_pct'].quantile(0.95)),
    'raw_exact_switch_pct': float(pp_compare_df['raw_exact'].mean() * 100),
    'guarded_exact_switch_pct': float(pp_compare_df['guarded_exact'].mean() * 100),
    'raw_switch_overlap_pct': float(pp_compare_df['raw_switch_overlap'].mean() * 100),
    'guarded_switch_overlap_pct': float(pp_compare_df['guarded_switch_overlap'].mean() * 100),
    'fallback_rate_pct': float(pp_compare_df['used_fallback'].mean() * 100),
    'comparison_pass': comparison_pass,
}
pp_compare_df.to_hdf(DIAG_FILE, key='pandapower_comparison', mode='a')
pd.DataFrame([PP_COMPARISON_SUMMARY]).to_hdf(DIAG_FILE, key='pandapower_summary', mode='a')

fig_pp = make_subplots(
    rows=1, cols=3,
    subplot_titles=['BFS vs pandapower teacher loss', 'Guarded GT pandapower loss gap', 'Switch overlap']
)
fig_pp.add_trace(go.Scatter(
    x=pp_compare_df['teacher_bfs_loss_mw'], y=pp_compare_df['teacher_pp_loss_mw'],
    mode='markers', name='teacher', marker=dict(size=6, opacity=0.7)
), row=1, col=1)
max_loss = max(pp_compare_df['teacher_bfs_loss_mw'].max(), pp_compare_df['teacher_pp_loss_mw'].max())
fig_pp.add_trace(go.Scatter(x=[0, max_loss], y=[0, max_loss], mode='lines', name='1:1', line=dict(dash='dash')), row=1, col=1)
fig_pp.add_trace(go.Histogram(x=pp_compare_df['guarded_pp_loss_gap_pct'], nbinsx=40, name='guarded gap %'), row=1, col=2)
fig_pp.add_vline(x=ACCEPT_LOSS_GAP_PCT, line_dash='dash', line_color='red', row=1, col=2)
fig_pp.add_trace(go.Box(y=pp_compare_df['raw_switch_overlap'] * 100, name='raw GT'), row=1, col=3)
fig_pp.add_trace(go.Box(y=pp_compare_df['guarded_switch_overlap'] * 100, name='guarded'), row=1, col=3)
fig_pp.update_layout(title='Pandapower Validation and Switching Comparison', template='plotly_white', height=470, width=1250)
fig_pp.update_xaxes(title_text='BFS loss (MW)', row=1, col=1)
fig_pp.update_yaxes(title_text='Pandapower loss (MW)', row=1, col=1)
fig_pp.update_xaxes(title_text='Loss gap (%)', row=1, col=2)
fig_pp.update_yaxes(title_text='Switch overlap (%)', row=1, col=3)
fig_pp.show()

print('═'*78)
print('Pandapower Comparative Analysis')
print('═'*78)
print(f'  Cases compared                  : {len(pp_compare_df)}  (skipped={pp_skipped})')
print(f'  BFS vs pandapower loss diff     : mean={PP_COMPARISON_SUMMARY["pp_vs_bfs_loss_diff_mean_pct"]:.3f}%  '
      f'p95={PP_COMPARISON_SUMMARY["pp_vs_bfs_loss_diff_p95_pct"]:.3f}%')
print(f'  Raw GT vs BFS teacher loss gap  : mean={PP_COMPARISON_SUMMARY["raw_gt_bfs_loss_gap_mean_pct"]:.3f}%')
print(f'  Guarded GT vs pandapower gap    : mean={PP_COMPARISON_SUMMARY["guarded_pp_loss_gap_mean_pct"]:.3f}%  '
      f'p95={PP_COMPARISON_SUMMARY["guarded_pp_loss_gap_p95_pct"]:.3f}%')
print(f'  Raw / guarded exact switching   : {PP_COMPARISON_SUMMARY["raw_exact_switch_pct"]:.1f}% / '
      f'{PP_COMPARISON_SUMMARY["guarded_exact_switch_pct"]:.1f}%')
print(f'  Raw / guarded switch overlap    : {PP_COMPARISON_SUMMARY["raw_switch_overlap_pct"]:.1f}% / '
      f'{PP_COMPARISON_SUMMARY["guarded_switch_overlap_pct"]:.1f}%')
print(f'  Guard fallback rate             : {PP_COMPARISON_SUMMARY["fallback_rate_pct"]:.1f}%')
print(f'  Acceptance target               : mean guarded loss gap <= {ACCEPT_LOSS_GAP_PCT:.1f}%')
print(f'  Verdict                         : {"PASS" if comparison_pass else "REVIEW / TRAIN LONGER"}')
print('═'*78)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 18 — FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════
summary = pd.read_hdf(DIAG_FILE, key='summary').iloc[0]

try:
    weights = pd.read_hdf(DIAG_FILE, key='loss_weights').iloc[0]
    lam_p   = weights['lambda_physics']
    lam_v   = weights['lambda_voltage']
except:
    lam_p, lam_v = 0.10, 0.50

try:
    gt_summary = pd.read_hdf(DIAG_FILE, key='gt_training_summary').iloc[0]
except:
    gt_summary = None

try:
    pp_summary = pd.read_hdf(DIAG_FILE, key='pandapower_summary').iloc[0]
except:
    pp_summary = None

print('═'*70)
print('DNR GT + META-LEARNING PIPELINE COMPLETE')
print('═'*70)
print()
print('Network')
print(f'  IEEE 33-bus, {N_LINES} lines, {N_OPEN} tie lines')
print(f'  Z_base = {Z_BASE:.4f} Ω  (impedances converted to pu)')
print(f'  Valid spanning trees: {N_TOPOS:,}')
print()
print('Diagnostic Results')
print(f'  Scenarios run   : {int(summary.n_scenarios):,}')
print(f'  Feasible        : {int(summary.n_feasible):,}  ({summary.n_feasible/summary.n_scenarios*100:.1f}%)')
print(f'  Topology classes: {int(summary.n_classes)}')
print(f'  Loss mean±std   : {summary.loss_mean:.5f} ± {summary.loss_std:.5f} MW')
print(f'  V_min mean      : {summary.vmin_mean:.4f} pu')
print()
print('Training Config')
print(f'  Model           : Structural GraphTransformer ({GT_LAYERS} layers, {GT_HIDDEN} hidden, {GT_HEADS} heads)')
print(f'  Meta-learning   : Reptile ({GT_META_EPISODES} episodes, {GT_INNER_STEPS} inner steps)')
print(f'  Loss            : CE + {GT_PHYS_LAMBDA:.2f}*physics  (diagnostic λ≈{lam_p:.2f}, voltage λ≈{lam_v:.2f})')
print(f'  Radial guard    : GT top-{GT_CANDIDATE_K} candidates + BFS validation')
print()
if gt_summary is not None:
    print('GT Validation')
    print(f'  Line F1         : {gt_summary.val_line_f1:.3f}')
    print(f'  Exact topology  : {gt_summary.val_exact_topology*100:.1f}%')
    print(f'  Switch overlap  : {gt_summary.val_switch_overlap*100:.1f}%')
    print(f'  Loss MAPE       : {gt_summary.val_loss_mape_pct:.2f}%')
    print()
if pp_summary is not None:
    verdict = 'PASS' if bool(pp_summary.comparison_pass) else 'REVIEW / TRAIN LONGER'
    print('Pandapower Comparison')
    print(f'  Cases compared  : {int(pp_summary.n_cases)}')
    print(f'  BFS↔PP loss diff: mean={pp_summary.pp_vs_bfs_loss_diff_mean_pct:.3f}%  p95={pp_summary.pp_vs_bfs_loss_diff_p95_pct:.3f}%')
    print(f'  Guarded GT gap  : mean={pp_summary.guarded_pp_loss_gap_mean_pct:.3f}%  p95={pp_summary.guarded_pp_loss_gap_p95_pct:.3f}%')
    print(f'  Switch overlap  : {pp_summary.guarded_switch_overlap_pct:.1f}%')
    print(f'  Fallback rate   : {pp_summary.fallback_rate_pct:.1f}%')
    print(f'  Verdict         : {verdict}')
    print()
print('Stored Data')
print(f'  File    : {DIAG_FILE}')
print('  Keys    : scenarios, summary, training_matrix_check, loss_weights,')
print('            gt_training_summary, gt_training_history,')
print('            pandapower_comparison, pandapower_summary')
print('═'*70)